In [ ]:
import pandas as pd
import numpy as np

# ====== 경로 ======
FUZZY_BEST_PATH = "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/5Fuzzy/best_all_sites_DOY2.csv"
OBS_PATH        = "/Users/doyoung-gil/연구실/데이터/사과/복숭아순나방_APPLE_2013_2024.csv"
GDD_PATH        = "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/joined_SAMPLE_GDD_2013_2024_APPLE_복숭아순나방.csv"
OUT_PATH        = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG.csv"

# ====== 컬럼명 ======
OBS_SITE_COL = "지점ID"
DATE_COL     = "조사일자(YYYYMMDD)"
YEAR_COL     = "조사년도"
FUZZY_SITE_COL = "site_id"

# ====== 1) fuzzy(best) ======
fuzzy = pd.read_csv(FUZZY_BEST_PATH, encoding="utf-8-sig", low_memory=False, dtype={FUZZY_SITE_COL: str})
fuzzy[FUZZY_SITE_COL] = fuzzy[FUZZY_SITE_COL].astype(str).str.strip()
fuzzy["_join_site"] = fuzzy[FUZZY_SITE_COL]

num_cols = [
    "dormancy_start_date","dormancy_end_date",
    "growing_start_date","growing_end_date",
    "flowering_date",
    "dormancy_months","growing_months","window_idx","offset_days"
]
for c in num_cols:
    if c in fuzzy.columns:
        fuzzy[c] = pd.to_numeric(fuzzy[c], errors="coerce")

fuzzy["year"] = pd.to_numeric(fuzzy["year"], errors="coerce").astype("Int64")
fuzzy = fuzzy[fuzzy["year"].notna()].copy()
fuzzy["year"] = fuzzy["year"].astype(int)

# ====== 2) 관측 데이터 ======
obs = pd.read_csv(OBS_PATH, encoding="utf-8-sig", low_memory=False)
obs[OBS_SITE_COL] = obs[OBS_SITE_COL].astype(str).str.strip()
obs[DATE_COL] = pd.to_datetime(obs[DATE_COL], errors="coerce").dt.normalize()
obs = obs[obs[DATE_COL].notna()].copy()

if YEAR_COL in obs.columns:
    obs[YEAR_COL] = pd.to_numeric(obs[YEAR_COL], errors="coerce")
    obs["year"] = obs[YEAR_COL].where(obs[YEAR_COL].notna(), obs[DATE_COL].dt.year)
else:
    obs["year"] = obs[DATE_COL].dt.year

obs["year"] = obs["year"].astype(int)
obs["obs_doy"] = obs[DATE_COL].dt.dayofyear.astype(int)

moth_cols = [c for c in obs.columns if "복숭아순나방" in c]
keep_obs = [OBS_SITE_COL, "year", DATE_COL, "obs_doy"] + moth_cols
obs2 = obs[keep_obs].copy()

# ====== 3) fuzzy + 관측 조인 ======
df = fuzzy.merge(
    obs2,
    left_on=["_join_site", "year"],
    right_on=[OBS_SITE_COL, "year"],
    how="inner"
)

# ====== 4) DOY 기반 phenology 파생 ======
xx = pd.to_numeric(df["obs_doy"], errors="coerce")

df["days_since_flowering"] = (xx - pd.to_numeric(df["flowering_date"], errors="coerce")).round()
df["days_since_growing_start"] = (xx - pd.to_numeric(df["growing_start_date"], errors="coerce")).round()
df["days_until_growing_end"] = (pd.to_numeric(df["growing_end_date"], errors="coerce") - xx).round()

df["is_growing"] = (
    (xx >= df["growing_start_date"]) & (xx <= df["growing_end_date"])
).fillna(False).astype(int)

df["is_dormancy"] = (
    ((df["dormancy_start_date"] <= df["dormancy_end_date"]) &
     (xx >= df["dormancy_start_date"]) & (xx <= df["dormancy_end_date"]))
    |
    ((df["dormancy_start_date"] > df["dormancy_end_date"]) &
     ((xx >= df["dormancy_start_date"]) | (xx <= df["dormancy_end_date"])))
).fillna(False).astype(int)

# ====== 5) 🔥 GDD 시계열 로드 & 조인 ======
gdd = pd.read_csv(GDD_PATH, encoding="utf-8-sig", low_memory=False)
gdd["지점ID"] = gdd["지점ID"].astype(str).str.strip()
gdd["일시"] = pd.to_datetime(gdd["일시"], errors="coerce").dt.normalize()

gdd = gdd[["지점ID", "일시", "GDD10_cum"]].copy()

df = df.merge(
    gdd,
    left_on=[OBS_SITE_COL, DATE_COL],
    right_on=["지점ID", "일시"],
    how="left"
)

# ====== 6) 컬럼 정리 ======
TRAP_COL = "(트랩)복숭아순나방-마리수"

df["label_event"] = (
    pd.to_numeric(df[TRAP_COL], errors="coerce").fillna(0) > 0
).astype(int)

# label_event 바로 왼쪽에 GDD10_cum 위치
cols = list(df.columns)
cols.remove("GDD10_cum")
label_idx = cols.index("label_event")
cols.insert(label_idx, "GDD10_cum")

# 불필요한 조인 컬럼 제거
drop_cols = ["_join_site", OBS_SITE_COL, "지점ID", "일시"]
cols = [c for c in cols if c not in drop_cols]

df_out = df[cols].copy()

# ====== 7) 저장 ======
print("[CHECK] rows:", len(df_out), "cols:", df_out.shape[1])
print(df_out.head(5))

df_out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("[SAVED]", OUT_PATH)


In [ ]:
# 전체 발생 비율
df_out["label_event"].value_counts(normalize=True)

In [ ]:
# 생육기 vs 비생육기 발생률
df_out.groupby("is_growing")["label_event"].mean()

In [ ]:
# 개화 기준 시점별 발생률(대충)
df_out.groupby(pd.cut(df_out["days_since_flowering"], [-100, -30, 0, 30, 60, 120]))["label_event"].mean()


In [ ]:
# ---- 안전 처리 ----
df = df_out.copy()

# suitability (0~1 클립)
S = df["best_suitability"].astype(float).clip(0, 1)

# phenology gate (소프트 게이트 권장)
# 비생육기에도 출현이 있으므로 0.4 vs 1.0 가중
G = df["is_growing"].astype(float).fillna(0)
G = np.where(G == 1, 1.0, 0.4)


d = df.get("days_since_flowering").astype(float)
# 시점 가중치
d = df.get("days_since_flowering").astype(float)
theta = 0   
k = 12       
W = 1 / (1 + np.exp(-(d - theta) / k))

df["risk_S"]     = 100 * (S)       * G * W      
df["risk_1mS"]   = 100 * (1 - S)   * G * W      # 가설 B: 취약성(1-S)↑ → 위험↑



In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

y = df["label_event"].astype(int)

auc_S   = roc_auc_score(y, df["risk_S"])
auc_1mS = roc_auc_score(y, df["risk_1mS"])

print(f"AUC (Risk = S):     {auc_S:.3f}")
print(f"AUC (Risk = 1-S):   {auc_1mS:.3f}")


In [ ]:
import matplotlib.pyplot as plt

fpr_S,   tpr_S,   _ = roc_curve(y, df["risk_S"])
fpr_1mS, tpr_1mS, _ = roc_curve(y, df["risk_1mS"])

plt.figure(figsize=(6,6))
plt.plot(fpr_S,   tpr_S,   label=f"Risk=S (AUC={auc_S:.3f})")
plt.plot(fpr_1mS, tpr_1mS, label=f"Risk=1-S (AUC={auc_1mS:.3f})")
plt.plot([0,1],[0,1],'k--',alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC: Risk score comparison")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def best_threshold(y_true, score):
    fpr, tpr, thr = roc_curve(y_true, score)
    j = tpr - fpr
    i = np.argmax(j)
    return thr[i], tpr[i], 1-fpr[i]

T_S,   TPR_S,   TNR_S   = best_threshold(y, df["risk_S"])
T_1mS, TPR_1mS, TNR_1mS = best_threshold(y, df["risk_1mS"])

print(f"[Risk=S]   T={T_S:.2f},   TPR={TPR_S:.2f}, TNR={TNR_S:.2f}")
print(f"[Risk=1-S] T={T_1mS:.2f}, TPR={TPR_1mS:.2f}, TNR={TNR_1mS:.2f}")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# =========================
# 0) 설정
# =========================
LABEL_COL = "label_event"

# GDD 포함 feature set
BASE_FEATURES = [
    "best_suitability",
    "is_growing",
    "days_since_flowering",
    "GDD10_cum",          
]

OPTIONAL_FEATURES = [
    "is_dormancy",
    "days_since_growing_start",
    "days_until_growing_end",
]

features = [c for c in (BASE_FEATURES + OPTIONAL_FEATURES) if c in df_out.columns]
print("[INFO] Using features:", features)

df_ml = df_out.copy()
df_ml = df_ml[df_ml[LABEL_COL].notna()].copy()

X = df_ml[features].apply(pd.to_numeric, errors="coerce")
y = df_ml[LABEL_COL].astype(int)


# =========================
# 1) Train / Test split (연도 기준)
# =========================
if "year" in df_ml.columns:
    years = np.sort(df_ml["year"].unique())
    n_test_years = max(1, int(np.ceil(len(years) * 0.2)))
    test_years = set(years[-n_test_years:])

    train_idx = ~df_ml["year"].isin(test_years)
    test_idx  = df_ml["year"].isin(test_years)

    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    print(f"[SPLIT] Year-based split, test years = {sorted(test_years)}")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print("[SPLIT] Random stratified split")

print("[SPLIT] train:", len(X_train), "test:", len(X_test))
print("[SPLIT] y_train mean:", y_train.mean(), "y_test mean:", y_test.mean())


# =========================
# 2) Logistic Regression (Baseline + GDD)
# =========================
logit = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])

logit.fit(X_train, y_train)
p_logit = logit.predict_proba(X_test)[:, 1]

auc_logit = roc_auc_score(y_test, p_logit)
ap_logit  = average_precision_score(y_test, p_logit)

print("\n=== Logistic Regression (with GDD) ===")
print("ROC-AUC:", round(auc_logit, 3))
print("PR-AUC :", round(ap_logit, 3))




# =========================
# 4) 요약
# =========================
summary = pd.DataFrame({
    "Model": [
        "Logistic (with GDD)",
        "RandomForest (with GDD)"
    ],
    "ROC_AUC": [auc_logit, auc_rf],
    "PR_AUC":  [ap_logit,  ap_rf]
})

print("\n=== SUMMARY (GDD added) ===")
print(summary.sort_values("ROC_AUC", ascending=False).to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# --- 필수 컬럼 체크 ---
need_cols = ["best_suitability", "is_growing", "days_since_flowering"]
missing = [c for c in need_cols if c not in df_out.columns]
if missing:
    raise ValueError(f"df_out에 필요한 컬럼이 없습니다: {missing}")

df = df_out.copy()

# 1) suitability (0~1)
S = pd.to_numeric(df["best_suitability"], errors="coerce").clip(0, 1)

# 2) phenology gate (소프트 게이트)
G_raw = pd.to_numeric(df["is_growing"], errors="coerce").fillna(0)
G = np.where(G_raw > 0, 1.0, 0.4)   # 생육기=1.0, 비생육기=0.4

# 3) days since flowering
d = pd.to_numeric(df["days_since_flowering"], errors="coerce")

# 4) 시점 가중치 sigmoid
theta = 0.0   # 위험 시작점(개화일 기준)
k     = 12.0  # 완만함
W = sigmoid((d - theta) / k)

# 5) Risk score (0~100 스케일)
df_out["risk_S"]   = (100.0 * S * G * W).astype(float)        # 가설 A
df_out["risk_1mS"] = (100.0 * (1.0 - S) * G * W).astype(float) # 가설 B

print("[OK] risk_S / risk_1mS created in df_out")
print(df_out[["risk_S", "risk_1mS"]].describe())


In [ ]:
# =========================
# 0) 설정
# =========================
LABEL_COL = "label_event"
YEAR_COL  = "year"

BASE_FEATURES = [
    "best_suitability",
    "is_growing",
    "days_since_flowering",
    "is_dormancy",
    "days_since_growing_start",
    "days_until_growing_end",
]

OPTIONAL_GDD_FEATURES = []
for c in ["GDD10_cum", "GDD_7d", "GDD_14d"]:
    if c in df_out.columns:
        OPTIONAL_GDD_FEATURES.append(c)


# =========================
# 1) (이전 로직) Sigmoid risk score 만들기
#    - S: 0~1 clip
#    - G: 생육기=1.0, 비생육기=0.4 (소프트 게이트)
#    - W: sigmoid((d-theta)/k)
#    - risk_S, risk_1mS: 0~100 스케일
# =========================
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

df = df_out.copy()

need_cols = ["best_suitability", "is_growing", "days_since_flowering", LABEL_COL, YEAR_COL]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise ValueError(f"df_out에 필요한 컬럼이 없습니다: {missing}")

S = pd.to_numeric(df["best_suitability"], errors="coerce").clip(0, 1)

G_raw = pd.to_numeric(df["is_growing"], errors="coerce").fillna(0)
G = np.where(G_raw > 0, 1.0, 0.4)   # ★ 소프트 게이트 (이전 로직)

d = pd.to_numeric(df["days_since_flowering"], errors="coerce")

theta = 0.0    
k     = 12.0    
W = sigmoid((d - theta) / k)

df["risk_S"]   = (100.0 * S * G * W).astype(float)
df["risk_1mS"] = (100.0 * (1.0 - S) * G * W).astype(float)

print("[CHECK] risk_S stats")
print(df["risk_S"].describe())
print("[CHECK] risk_1mS stats")
print(df["risk_1mS"].describe())


# =========================
# 2) Train/Test split (연도 기준)
# =========================
df_ml = df[df[LABEL_COL].notna()].copy()
y = df_ml[LABEL_COL].astype(int)

years = np.sort(df_ml[YEAR_COL].unique())
n_test_years = max(1, int(np.ceil(len(years) * 0.2)))
test_years = set(years[-n_test_years:])

train_idx = ~df_ml[YEAR_COL].isin(test_years)
test_idx  = df_ml[YEAR_COL].isin(test_years)

y_train, y_test = y.loc[train_idx], y.loc[test_idx]
print(f"\n[SPLIT] test years = {sorted(test_years)}")
print("[SPLIT] train:", int(train_idx.sum()), "test:", int(test_idx.sum()))
print("[SPLIT] y_train mean:", y_train.mean(), "y_test mean:", y_test.mean())


# =========================
# 3) (A) Risk score 단독 성능(ROC/PR)
# =========================
riskS_test = pd.to_numeric(df_ml.loc[test_idx, "risk_S"], errors="coerce").fillna(0).values
risk1mS_test = pd.to_numeric(df_ml.loc[test_idx, "risk_1mS"], errors="coerce").fillna(0).values

auc_riskS = roc_auc_score(y_test, riskS_test)
ap_riskS  = average_precision_score(y_test, riskS_test)

auc_risk1mS = roc_auc_score(y_test, risk1mS_test)
ap_risk1mS  = average_precision_score(y_test, risk1mS_test)

print("\n=== Risk Score only ===")
print("risk_S   ROC-AUC:", round(auc_riskS, 3), " PR-AUC:", round(ap_riskS, 3))
print("risk_1mS ROC-AUC:", round(auc_risk1mS, 3), " PR-AUC:", round(ap_risk1mS, 3))


# =========================
# 4) (B) Hybrid ML: baseline + risk_S (+optional gdd)
# =========================
features = [c for c in BASE_FEATURES if c in df_ml.columns] + OPTIONAL_GDD_FEATURES + ["risk_S"]
features = list(dict.fromkeys(features))  # 중복 제거
print("\n[INFO] Hybrid features:", features)

X = df_ml[features].apply(pd.to_numeric, errors="coerce")
X_train, X_test = X.loc[train_idx], X.loc[test_idx]


# ---- Logistic ----
logit = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])
logit.fit(X_train, y_train)
p_logit = logit.predict_proba(X_test)[:, 1]

auc_logit = roc_auc_score(y_test, p_logit)
ap_logit  = average_precision_score(y_test, p_logit)

print("\n=== Logistic Regression (Hybrid: +risk_S) ===")
print("ROC-AUC:", round(auc_logit, 3))
print("PR-AUC :", round(ap_logit, 3))


# ---- Random Forest ----
rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
p_rf = rf.predict_proba(X_test)[:, 1]

auc_rf = roc_auc_score(y_test, p_rf)
ap_rf  = average_precision_score(y_test, p_rf)

print("\n=== Random Forest (Hybrid: +risk_S) ===")
print("ROC-AUC:", round(auc_rf, 3))
print("PR-AUC :", round(ap_rf, 3))


# =========================
# 5) 요약
# =========================
summary = pd.DataFrame({
    "Model": [
        "Risk_S only",
        "Risk_1mS only",
        "Logistic (Hybrid +risk_S)",
        "RF (Hybrid +risk_S)"
    ],
    "ROC_AUC": [auc_riskS, auc_risk1mS, auc_logit, auc_rf],
    "PR_AUC":  [ap_riskS,  ap_risk1mS,  ap_logit,  ap_rf],
})
print("\n=== SUMMARY ===")
print(summary.sort_values("ROC_AUC", ascending=False).to_string(index=False))


In [ ]:

# =========================
# 0) 설정
# =========================
LABEL_COL = "label_event"
YEAR_COL  = "year"

# df_out에 이미 만들어둔 risk_S를 쓸 거야.
# (없으면 이전에 만든 sigmoid 블록 먼저 실행해서 df_out["risk_S"] 만들어둬야 함)
if "risk_S" not in df_out.columns:
    raise ValueError("df_out에 'risk_S' 컬럼이 없습니다. 먼저 risk_S 생성 코드를 실행하세요.")

df = df_out.copy()
df = df[df[LABEL_COL].notna()].copy()


# =========================
# 1) Split (연도 기준: 최근 20% 연도)
# =========================
years = np.sort(df[YEAR_COL].unique())
n_test_years = max(1, int(np.ceil(len(years) * 0.2)))
test_years = set(years[-n_test_years:])

train_idx = ~df[YEAR_COL].isin(test_years)
test_idx  = df[YEAR_COL].isin(test_years)

y_train = df.loc[train_idx, LABEL_COL].astype(int)
y_test  = df.loc[test_idx,  LABEL_COL].astype(int)

print(f"[SPLIT] test years = {sorted(test_years)}")
print("[SPLIT] train:", int(train_idx.sum()), "test:", int(test_idx.sum()))
print("[SPLIT] y_train mean:", y_train.mean(), "y_test mean:", y_test.mean())


# =========================
# 2) Feature sets (baseline 완전 제거)
# =========================
feature_sets = []

# (A) risk_S only
feature_sets.append(("risk_S only", ["risk_S"]))

# (B) risk_S + GDD 누적(있으면)
if "GDD10_cum" in df.columns:
    feature_sets.append(("risk_S + GDD10_cum", ["risk_S", "GDD10_cum"]))

# (C) risk_S + GDD 단기(있으면)
gdd_short = [c for c in ["GDD_7d", "GDD_14d"] if c in df.columns]
if len(gdd_short) > 0:
    feature_sets.append((f"risk_S + {','.join(gdd_short)}", ["risk_S"] + gdd_short))

print("\n[INFO] Feature sets:")
for name, feats in feature_sets:
    print(" -", name, ":", feats)


# =========================
# 3) 모델 함수
# =========================
def eval_models(X_train, X_test, y_train, y_test):
    out = []

    # Logistic
    logit = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000))
    ])
    logit.fit(X_train, y_train)
    p_logit = logit.predict_proba(X_test)[:, 1]
    out.append((
        "Logistic",
        roc_auc_score(y_test, p_logit),
        average_precision_score(y_test, p_logit)
    ))


    return out


# =========================
# 4) 돌리기
# =========================
rows = []
for fs_name, feats in feature_sets:
    X = df[feats].apply(pd.to_numeric, errors="coerce")
    X_train = X.loc[train_idx]
    X_test  = X.loc[test_idx]

    # (참고) risk_S 단독도 같이 찍어두면 비교가 쉬움
    # risk_S 자체를 점수로 쓸 때의 성능
    if feats == ["risk_S"]:
        score = pd.to_numeric(df.loc[test_idx, "risk_S"], errors="coerce").fillna(0).values
        auc_score = roc_auc_score(y_test, score)
        ap_score  = average_precision_score(y_test, score)
        rows.append([fs_name, "RiskScore-as-score", auc_score, ap_score])

    for model_name, auc, ap in eval_models(X_train, X_test, y_train, y_test):
        rows.append([fs_name, model_name, auc, ap])

result = pd.DataFrame(rows, columns=["FeatureSet", "Model", "ROC_AUC", "PR_AUC"])
print("\n=== RESULTS ===")
print(result.sort_values(["ROC_AUC"], ascending=False).to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# =========================
# 0) 필수 컬럼 체크
# =========================
need = ["best_suitability", "is_growing", "days_since_flowering", "label_event", "year"]
missing = [c for c in need if c not in df_out.columns]
if missing:
    raise ValueError(f"df_out에 필요한 컬럼이 없습니다: {missing}")

df = df_out.copy()
df = df[df["label_event"].notna()].copy()

# 숫자형 정리
S = pd.to_numeric(df["best_suitability"], errors="coerce").clip(0, 1)
G = (pd.to_numeric(df["is_growing"], errors="coerce").fillna(0) > 0).astype(float)
d = pd.to_numeric(df["days_since_flowering"], errors="coerce")
y = df["label_event"].astype(int)

# =========================
# 1) 연도 기반 split (너랑 동일)
# =========================
years = np.sort(df["year"].unique())
n_test_years = max(1, int(np.ceil(len(years) * 0.2)))
test_years = set(years[-n_test_years:])

test_idx = df["year"].isin(test_years)
y_test = y.loc[test_idx]

print("[SPLIT] test years =", sorted(test_years))
print("[SPLIT] test size =", int(test_idx.sum()))

# =========================
# 2) sigmoid / risk 함수
# =========================
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def make_risk(theta, k):
    # k가 너무 작으면 overflow 날 수 있어 안전장치
    k = float(k)
    z = (d - float(theta)) / k
    W = sigmoid(z)
    risk = (S * G * W).astype(float)
    # 결측은 0으로 (평가에서 제외하고 싶으면 dropna로 바꿔도 됨)
    return risk.fillna(0.0)

# =========================
# 3) 그리드 설정
#   - theta: 위험 상승 시작점 (days since flowering)
#   - k: 완만함
# =========================
theta_grid = list(range(-30, 91, 5))   # -30 ~ 90, 5일 간격
k_grid     = [3, 5, 7, 10, 12, 15, 20] # 완만함 후보

# (선택) 관측 범위에 맞춰 자동으로 theta 범위를 잡고 싶으면:
# tmin = int(np.nanpercentile(d, 1))
# tmax = int(np.nanpercentile(d, 99))
# theta_grid = list(range((tmin//5)*5, (tmax//5)*5 + 1, 5))

# =========================
# 4) Grid Search
# =========================
rows = []
best = {"roc": -1, "theta": None, "k": None, "pr": None}

for theta in theta_grid:
    for k in k_grid:
        risk = make_risk(theta, k)
        score_test = risk.loc[test_idx].values

        # ROC-AUC 계산
        roc = roc_auc_score(y_test, score_test)
        pr  = average_precision_score(y_test, score_test)

        rows.append([theta, k, roc, pr])

        if roc > best["roc"]:
            best.update({"roc": roc, "theta": theta, "k": k, "pr": pr})

res = pd.DataFrame(rows, columns=["theta", "k", "ROC_AUC", "PR_AUC"])
res = res.sort_values("ROC_AUC", ascending=False).reset_index(drop=True)

print("\n=== BEST (by ROC-AUC) ===")
print(f"theta={best['theta']}, k={best['k']}, ROC-AUC={best['roc']:.3f}, PR-AUC={best['pr']:.3f}")

print("\n=== TOP 15 ===")
print(res.head(15).to_string(index=False))

# =========================
# 5) 최적 risk_S를 df_out에 저장(원하면)
# =========================
theta_star, k_star = best["theta"], best["k"]
df_out["risk_S_opt"] = make_risk(theta_star, k_star).values
print("\n[OK] df_out['risk_S_opt'] created with best theta/k")


In [ ]:
# 지점별 label_event 평균과 표본수
site_stats = df_out.groupby("site_id")["label_event"].agg(
    n="count",
    event_rate="mean",
    n_pos="sum"
).sort_values(["event_rate","n"], ascending=[True, False])

# 아예 1이 한번도 없는 지점
zero_pos_sites = site_stats[site_stats["n_pos"] == 0]

print("총 지점 수:", site_stats.shape[0])
print("양성(1) 0회 지점 수:", zero_pos_sites.shape[0])
print(zero_pos_sites.head(20))


### 휴먼기 이후부터 GDD 계산

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =======================
# 설정
# =======================
BEST_PATH  = Path("/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/5Fuzzy/best_all_sites_DOY2.csv")
NEIGH_PATH = Path("/Users/doyoung-gil/연구실/데이터/사과/neighbors_master.csv")

WEATHER_DIR  = Path("/Users/doyoung-gil/연구실/데이터/기상데이터/ASOS+AWS")
WEATHER_GLOB = "ASOS_AWS_*.csv"

OUT_DIR = Path("/Users/doyoung-gil/연구실/데이터/사과/R_GDD")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SITE_COL = "site_id"
YEAR_COL = "year"
DOY_COL  = "dormancy_end_date"   # DOY (연도별)

# neighbors_master 컬럼(예전 네 포맷)
NEI_SITE_COL = "지점ID"
NEI_STN_COL  = "지점"
NEI_RANK_COL = "rank"
NEI_W_COL    = "w_raw"

# weather 파일 컬럼
WX_STN_COL   = "지점"
WX_DATE_COL  = "일시"
WX_TMEAN_COL = "평균기온(°C)"    # 없으면 아래 후보/대체로 확장 가능

# GDD 설정
TBASE = 10.0
K_NEIGH = 3  # neighbors_master에서 rank 1..3 사용

# =======================
# 함수
# =======================
def read_csv_flexible(p: Path) -> pd.DataFrame:
    for enc in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
        try:
            return pd.read_csv(p, encoding=enc, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(p, low_memory=False)

def doy_to_date(year: int, doy: int) -> pd.Timestamp:
    return pd.Timestamp(year=year, month=1, day=1) + pd.Timedelta(days=int(doy)-1)

def load_weather_for_stations_years(stn_ids: list[int], years: list[int]) -> pd.DataFrame:
    """
    여러 관측소(stn_ids) + 여러 연도(years)의 일별 평균기온을 로드해서 반환
    return columns: [date, stn_id, Tmean]
    """
    stn_set = set(int(x) for x in stn_ids)
    year_set = set(int(y) for y in years)

    parts = []
    for p in sorted(WEATHER_DIR.glob(WEATHER_GLOB)):
        w = read_csv_flexible(p)
        if WX_DATE_COL not in w.columns or WX_STN_COL not in w.columns:
            continue

        w[WX_DATE_COL] = pd.to_datetime(w[WX_DATE_COL], errors="coerce").dt.normalize()
        w = w.dropna(subset=[WX_DATE_COL])
        if w.empty:
            continue

        # 연도 필터
        w = w[w[WX_DATE_COL].dt.year.isin(year_set)]
        if w.empty:
            continue

        w[WX_STN_COL] = pd.to_numeric(w[WX_STN_COL], errors="coerce")
        w = w.dropna(subset=[WX_STN_COL])
        if w.empty:
            continue

        w[WX_STN_COL] = w[WX_STN_COL].astype(int)
        w = w[w[WX_STN_COL].isin(stn_set)]
        if w.empty:
            continue

        if WX_TMEAN_COL not in w.columns:
            raise SystemExit(f"[에러] 기상 파일에 '{WX_TMEAN_COL}' 컬럼이 없습니다. "
                             f"대체 컬럼(최고/최저 등)로 만드는 로직을 추가해야 합니다.")
        w[WX_TMEAN_COL] = pd.to_numeric(w[WX_TMEAN_COL], errors="coerce")

        parts.append(w[[WX_DATE_COL, WX_STN_COL, WX_TMEAN_COL]])

    if not parts:
        return pd.DataFrame(columns=["date", "stn_id", "Tmean"])

    wx = (pd.concat(parts, ignore_index=True)
            .drop_duplicates(subset=[WX_DATE_COL, WX_STN_COL])
            .rename(columns={WX_DATE_COL: "date", WX_STN_COL: "stn_id", WX_TMEAN_COL: "Tmean"}))
    wx["stn_id"] = wx["stn_id"].astype(int)
    return wx.sort_values(["stn_id", "date"]).reset_index(drop=True)

def idw_site_tmean(mat: pd.DataFrame, stns: list[int], ws: np.ndarray) -> pd.Series:
    """
    mat: date x stn_id (Tmean pivot)
    stns: 이 site가 쓰는 관측소 id (len K)
    ws:  w_raw (len K)
    반환: date index를 가진 Tmean_idw Series
    """
    sub = mat.reindex(columns=stns)
    X = sub.to_numpy(dtype=float)           # (days, K)
    valid = np.isfinite(X)

    W = np.tile(ws, (X.shape[0], 1))
    W = np.where(valid, W, 0.0)

    denom = W.sum(axis=1)
    num = np.nansum(X * W, axis=1)

    tmean_idw = np.divide(num, denom, out=np.full_like(num, np.nan), where=(denom > 0))
    return pd.Series(tmean_idw, index=mat.index, name="Tmean_idw")

# =======================
# 메인
# =======================
best = read_csv_flexible(BEST_PATH)
best[SITE_COL] = best[SITE_COL].astype(str).str.strip()
best[YEAR_COL] = pd.to_numeric(best[YEAR_COL], errors="coerce").astype("Int64")
best[DOY_COL]  = pd.to_numeric(best[DOY_COL], errors="coerce").astype("Int64")

# neighbors_master 로드
neigh = read_csv_flexible(NEIGH_PATH)
need = {NEI_SITE_COL, NEI_STN_COL, NEI_RANK_COL, NEI_W_COL}
missing = need - set(neigh.columns)
if missing:
    raise SystemExit(f"[에러] neighbors_master.csv에 필수 컬럼이 없습니다: {missing}")

neigh[NEI_SITE_COL] = neigh[NEI_SITE_COL].astype(str).str.strip()
neigh[NEI_STN_COL]  = pd.to_numeric(neigh[NEI_STN_COL], errors="coerce").astype("Int64")
neigh[NEI_RANK_COL] = pd.to_numeric(neigh[NEI_RANK_COL], errors="coerce").astype("Int64")
neigh[NEI_W_COL]    = pd.to_numeric(neigh[NEI_W_COL], errors="coerce")

# rank 1..K만 사용
neigh = (neigh.dropna(subset=[NEI_SITE_COL, NEI_STN_COL, NEI_RANK_COL, NEI_W_COL])
              .query(f"{NEI_RANK_COL} >= 1 and {NEI_RANK_COL} <= @K_NEIGH")
              .sort_values([NEI_SITE_COL, NEI_RANK_COL])
              .drop_duplicates([NEI_SITE_COL, NEI_RANK_COL]))

# best에 등장하는 site만
site_set = set(best[SITE_COL].unique())
neigh = neigh[neigh[NEI_SITE_COL].isin(site_set)].copy()

# =======================
# site별 처리
# =======================
for site_id, g in best.groupby(SITE_COL):
    g = g.copy()
    if site_id == "":
        continue

    ng = neigh[neigh[NEI_SITE_COL] == site_id].sort_values(NEI_RANK_COL)
    if ng.empty:
        continue

    years = sorted(g[YEAR_COL].dropna().astype(int).unique().tolist())
    if not years:
        continue

    stn_ids = ng[NEI_STN_COL].astype(int).tolist()
    ws = ng[NEI_W_COL].astype(float).to_numpy()

    # 기상 로드 (해당 site에 필요한 관측소 + 연도만)
    wx = load_weather_for_stations_years(stn_ids, years)
    if wx.empty:
        continue

    # pivot date x stn
    mat = wx.pivot(index="date", columns="stn_id", values="Tmean").sort_index()

    # IDW 보강 Tmean
    tmean_idw = idw_site_tmean(mat, stn_ids, ws)

    out = pd.DataFrame({"date": tmean_idw.index, "Tmean_idw": tmean_idw.values})
    out["year"] = out["date"].dt.year
    dd10 = (out["Tmean_idw"] - TBASE).clip(lower=0)
    out["DD10"] = dd10.fillna(0.0)


    # year -> dormancy_end_date(date)
    db_map = {}
    for y, sub in g.groupby(YEAR_COL):
        if pd.isna(y):
            continue
        d = sub[DOY_COL].dropna()
        if d.empty:
            continue
        db_map[int(y)] = doy_to_date(int(y), int(d.iloc[0]))

    out["dormancy_end_date"] = out["year"].map(db_map)

    # 누적 GDD: DB 이전 0, DB 이후 누적 / 연도별 리셋
    def cum_since_db(sub):
        db = sub["dormancy_end_date"].iloc[0]
        gdd = np.zeros(len(sub), dtype=float)  # DB 이전도 0.0

        if pd.isna(db):
            sub["GDD10_since_db"] = np.nan
            return sub

        mask = sub["date"] >= db

        dd = sub["DD10"].to_numpy(dtype=float)
        dd_mask = dd[mask.to_numpy()]

        # ✅ 변경: 결측 증가량은 0으로 처리 -> 누적 끊김 방지
        dd_mask = np.nan_to_num(dd_mask, nan=0.0)

        gdd[mask.to_numpy()] = np.cumsum(dd_mask)
        sub["GDD10_since_db"] = gdd
        return sub


    out = out.groupby("year", group_keys=False).apply(cum_since_db)

    # ✅ GDD 소수점 1자리
    out["GDD10_since_db"] = out["GDD10_since_db"].round(1)

    # 저장
    out_path = OUT_DIR / f"site_{site_id}_GDD_timeseries.csv"
    out.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.1f")
    print(f"[저장] {out_path}")

print(f"[완료] 모든 사이트 결과가 저장됨 → {OUT_DIR}")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ====== 경로 ======
FUZZY_BEST_PATH = "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/5Fuzzy/best_all_sites_DOY2.csv"
OBS_PATH        = "/Users/doyoung-gil/연구실/데이터/사과/복숭아순나방_APPLE_2013_2024.csv"

RGDD_DIR        = Path("/Users/doyoung-gil/연구실/데이터/사과/R_GDD")
OUT_PATH        = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2.csv"

# ====== 컬럼명 ======
OBS_SITE_COL = "지점ID"
DATE_COL     = "조사일자(YYYYMMDD)"
YEAR_COL     = "조사년도"
FUZZY_SITE_COL = "site_id"

# ====== 1) fuzzy(best) ======
fuzzy = pd.read_csv(FUZZY_BEST_PATH, encoding="utf-8-sig", low_memory=False, dtype={FUZZY_SITE_COL: str})
fuzzy[FUZZY_SITE_COL] = fuzzy[FUZZY_SITE_COL].astype(str).str.strip()
fuzzy["_join_site"] = fuzzy[FUZZY_SITE_COL]

num_cols = [
    "dormancy_start_date","dormancy_end_date",
    "growing_start_date","growing_end_date",
    "flowering_date",
    "dormancy_months","growing_months","window_idx","offset_days"
]
for c in num_cols:
    if c in fuzzy.columns:
        fuzzy[c] = pd.to_numeric(fuzzy[c], errors="coerce")

fuzzy["year"] = pd.to_numeric(fuzzy["year"], errors="coerce").astype("Int64")
fuzzy = fuzzy[fuzzy["year"].notna()].copy()
fuzzy["year"] = fuzzy["year"].astype(int)

# ====== 2) 관측 데이터 ======
obs = pd.read_csv(OBS_PATH, encoding="utf-8-sig", low_memory=False)
obs[OBS_SITE_COL] = obs[OBS_SITE_COL].astype(str).str.strip()
obs[DATE_COL] = pd.to_datetime(obs[DATE_COL], errors="coerce").dt.normalize()
obs = obs[obs[DATE_COL].notna()].copy()

if YEAR_COL in obs.columns:
    obs[YEAR_COL] = pd.to_numeric(obs[YEAR_COL], errors="coerce")
    obs["year"] = obs[YEAR_COL].where(obs[YEAR_COL].notna(), obs[DATE_COL].dt.year)
else:
    obs["year"] = obs[DATE_COL].dt.year

obs["year"] = obs["year"].astype(int)
obs["obs_doy"] = obs[DATE_COL].dt.dayofyear.astype(int)

moth_cols = [c for c in obs.columns if "복숭아순나방" in c]
keep_obs = [OBS_SITE_COL, "year", DATE_COL, "obs_doy"] + moth_cols
obs2 = obs[keep_obs].copy()

# ====== 3) fuzzy + 관측 조인 ======
df = fuzzy.merge(
    obs2,
    left_on=["_join_site", "year"],
    right_on=[OBS_SITE_COL, "year"],
    how="inner"
)

# ====== 4) DOY 기반 phenology 파생 ======
xx = pd.to_numeric(df["obs_doy"], errors="coerce")

df["days_since_flowering"] = (xx - pd.to_numeric(df["flowering_date"], errors="coerce")).round()
df["days_since_growing_start"] = (xx - pd.to_numeric(df["growing_start_date"], errors="coerce")).round()
df["days_until_growing_end"] = (pd.to_numeric(df["growing_end_date"], errors="coerce") - xx).round()

df["is_growing"] = (
    (xx >= df["growing_start_date"]) & (xx <= df["growing_end_date"])
).fillna(False).astype(int)

df["is_dormancy"] = (
    ((df["dormancy_start_date"] <= df["dormancy_end_date"]) &
     (xx >= df["dormancy_start_date"]) & (xx <= df["dormancy_end_date"]))
    |
    ((df["dormancy_start_date"] > df["dormancy_end_date"]) &
     ((xx >= df["dormancy_start_date"]) | (xx <= df["dormancy_end_date"])))
).fillna(False).astype(int)

# ==========================================================
# 5) ✅ R_GDD site별 시계열에서 조사일자(date)에 맞는 GDD 붙이기
#     ✅ (중요) RGDD의 GDD10_since_db NA 전파를 ffill로 복구
# ==========================================================

need = df[[OBS_SITE_COL, DATE_COL]].dropna().drop_duplicates().copy()
need[OBS_SITE_COL] = need[OBS_SITE_COL].astype(str).str.strip()
need[DATE_COL] = pd.to_datetime(need[DATE_COL], errors="coerce").dt.normalize()
need = need[need[DATE_COL].notna()].copy()

dates_by_site = need.groupby(OBS_SITE_COL)[DATE_COL].apply(lambda s: set(s.tolist())).to_dict()

gdd_parts = []
for site_id, date_set in dates_by_site.items():
    p = RGDD_DIR / f"site_{site_id}_GDD_timeseries.csv"
    if not p.exists():
        continue

    g = pd.read_csv(p, encoding="utf-8-sig", low_memory=False)
    if "date" not in g.columns or "GDD10_since_db" not in g.columns:
        continue

    g["date"] = pd.to_datetime(g["date"], errors="coerce").dt.normalize()
    g = g[g["date"].notna()].copy()
    if g.empty:
        continue

    # ✅ date 정렬 후, 누적 GDD 결측을 전날 값으로 유지 (누적 끊김 방지)
    g = g.sort_values("date").reset_index(drop=True)
    g["GDD10_since_db"] = pd.to_numeric(g["GDD10_since_db"], errors="coerce")
    g["GDD10_since_db"] = g["GDD10_since_db"].ffill().fillna(0.0)

    # 필요한 날짜만 필터
    g = g[g["date"].isin(date_set)].copy()
    if g.empty:
        continue

    g["지점ID"] = str(site_id)
    gdd_parts.append(g[["지점ID", "date", "GDD10_since_db"]])

gdd_map = pd.concat(gdd_parts, ignore_index=True) if gdd_parts else pd.DataFrame(columns=["지점ID","date","GDD10_since_db"])

df = df.merge(
    gdd_map,
    left_on=[OBS_SITE_COL, DATE_COL],
    right_on=["지점ID", "date"],
    how="left"
)

# 소수점 1자리 유지
df["GDD10_since_db"] = pd.to_numeric(df["GDD10_since_db"], errors="coerce").round(1)

# ====== 6) 라벨 생성 + 컬럼 정리 ======
TRAP_COL = "(트랩)복숭아순나방-마리수"
df["label_event"] = (pd.to_numeric(df[TRAP_COL], errors="coerce").fillna(0) > 0).astype(int)

cols = list(df.columns)
if "GDD10_since_db" in cols:
    cols.remove("GDD10_since_db")
    label_idx = cols.index("label_event")
    cols.insert(label_idx, "GDD10_since_db")

drop_cols = ["_join_site", OBS_SITE_COL, "지점ID", "date"]
cols = [c for c in cols if c not in drop_cols]

df_out = df[cols].copy()

# ====== 7) 저장 ======
print("[CHECK] rows:", len(df_out), "cols:", df_out.shape[1])
print("GDD NA ratio:", df_out["GDD10_since_db"].isna().mean())
print(df_out.head(5))

df_out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("[SAVED]", OUT_PATH)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

LABEL_COL = "label_event"

# ✅ GDD 컬럼 자동 선택: cum 있으면 cum, 없으면 since_db
GDD_COL = "GDD10_since_db" 

BASE_FEATURES = [
    "best_suitability",
    "is_growing",
    "days_since_flowering",
]
if GDD_COL:
    BASE_FEATURES.append(GDD_COL)

OPTIONAL_FEATURES = [
    "is_dormancy",
    "days_since_growing_start",
    "days_until_growing_end",
]

features = [c for c in (BASE_FEATURES + OPTIONAL_FEATURES) if c in df_out.columns]
print("Using features:", features)

df_ml = df_out.copy()
df_ml = df_ml[df_ml[LABEL_COL].notna()].copy()

X = df_ml[features].apply(pd.to_numeric, errors="coerce")
y = df_ml[LABEL_COL].astype(int)

# ===== 1) Split (연도 기준) =====
if "year" in df_ml.columns:
    years = np.sort(df_ml["year"].unique())
    n_test_years = max(1, int(np.ceil(len(years) * 0.2)))
    test_years = set(years[-n_test_years:])

    train_idx = ~df_ml["year"].isin(test_years)
    test_idx  =  df_ml["year"].isin(test_years)

    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    print(f"Year-based split, test years = {sorted(test_years)}")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print("Random stratified split")

print("train:", len(X_train), "test:", len(X_test))
print("y_train mean:", y_train.mean(), "y_test mean:", y_test.mean())

# ===== 2) Logistic Regression =====
logit = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])
logit.fit(X_train, y_train)
p_logit = logit.predict_proba(X_test)[:, 1]

auc_logit = roc_auc_score(y_test, p_logit)
ap_logit  = average_precision_score(y_test, p_logit)

print("\n=== Logistic Regression ===")
print("ROC-AUC:", round(auc_logit, 3))
print("PR-AUC :", round(ap_logit, 3))

# ===== 3) RandomForest =====
rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])
rf.fit(X_train, y_train)
p_rf = rf.predict_proba(X_test)[:, 1]

auc_rf = roc_auc_score(y_test, p_rf)
ap_rf  = average_precision_score(y_test, p_rf)

print("\n=== RandomForest ===")
print("ROC-AUC:", round(auc_rf, 3))
print("PR-AUC :", round(ap_rf, 3))

# ===== 4) Summary =====
summary = pd.DataFrame({
    "Model": ["Logistic", "RandomForest"],
    "ROC_AUC": [auc_logit, auc_rf],
    "PR_AUC":  [ap_logit,  ap_rf],
})
print("\n=== SUMMARY ===")
print(summary.sort_values("ROC_AUC", ascending=False).to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# =========================================================
# 전제:
# - df_out 이 이미 메모리에 존재 (너가 만든 LONG2 로드 결과)
# - df_out에는 최소한: year, label_event, GDD10_since_db(또는 GDD10_cum),
#   best_suitability, is_growing, days_since_flowering ... 등이 들어있어야 함
# =========================================================

LABEL_COL = "label_event"
YEAR_COL  = "year"

# ✅ GDD 컬럼 자동 선택: cum 있으면 cum, 없으면 since_db
GDD_COL = "GDD10_cum" if "GDD10_cum" in df_out.columns else (
          "GDD10_since_db" if "GDD10_since_db" in df_out.columns else None)

if GDD_COL is None:
    raise ValueError("df_out에 GDD 컬럼이 없습니다. (GDD10_cum 또는 GDD10_since_db)")

# -----------------------
# Feature sets
# -----------------------
FEATURES_GDD_ONLY = [GDD_COL]

FEATURES_FULL = [
    "best_suitability",
    "is_growing",
    "days_since_flowering",
    GDD_COL,
    "is_dormancy",
    "days_since_growing_start",
    "days_until_growing_end",
]
FEATURES_FULL = [c for c in FEATURES_FULL if c in df_out.columns]  # 존재하는 것만

print("GDD_COL:", GDD_COL)
print("FEATURES_GDD_ONLY:", FEATURES_GDD_ONLY)
print("FEATURES_FULL:", FEATURES_FULL)


# =========================================================
# 공정 비교용: 동일 split, 동일 전처리, 동일 평가
# =========================================================
def year_based_split(df: pd.DataFrame, year_col: str, test_ratio: float = 0.2):
    years = np.sort(df[year_col].dropna().unique())
    n_test_years = max(1, int(np.ceil(len(years) * test_ratio)))
    test_years = set(years[-n_test_years:])
    train_idx = ~df[year_col].isin(test_years)
    test_idx  =  df[year_col].isin(test_years)
    return train_idx, test_idx, sorted(test_years)

def eval_model(df: pd.DataFrame, features: list[str], model_name: str):
    # label 결측 제거
    d = df[df[LABEL_COL].notna()].copy()

    # feature 숫자화
    X = d[features].apply(pd.to_numeric, errors="coerce")
    y = d[LABEL_COL].astype(int)

    # split (연도 기준)
    if YEAR_COL in d.columns:
        train_idx, test_idx, test_years = year_based_split(d, YEAR_COL, test_ratio=0.2)
        X_train, X_test = X.loc[train_idx], X.loc[test_idx]
        y_train, y_test = y.loc[train_idx], y.loc[test_idx]
        split_info = f"Year-based split, test years={test_years}"
    else:
        raise ValueError("df_out에 year 컬럼이 없습니다. 연도 기준 split 불가.")

    # 모델 정의 (Logit, RF 둘 다 평가)
    logit = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000))
    ])

    rf = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample"
        ))
    ])

    # Fit / Predict
    logit.fit(X_train, y_train)
    p_logit = logit.predict_proba(X_test)[:, 1]

    rf.fit(X_train, y_train)
    p_rf = rf.predict_proba(X_test)[:, 1]

    # Metrics
    auc_logit = roc_auc_score(y_test, p_logit)
    ap_logit  = average_precision_score(y_test, p_logit)

    auc_rf = roc_auc_score(y_test, p_rf)
    ap_rf  = average_precision_score(y_test, p_rf)

    out = pd.DataFrame([
        {"FeatureSet": model_name, "Model": "LogisticRegression", "ROC_AUC": auc_logit, "PR_AUC": ap_logit,
         "train_n": len(X_train), "test_n": len(X_test), "y_train_mean": y_train.mean(), "y_test_mean": y_test.mean(),
         "split": split_info},
        {"FeatureSet": model_name, "Model": "RandomForest", "ROC_AUC": auc_rf, "PR_AUC": ap_rf,
         "train_n": len(X_train), "test_n": len(X_test), "y_train_mean": y_train.mean(), "y_test_mean": y_test.mean(),
         "split": split_info},
    ])
    return out


# =========================================================
# 실행
# =========================================================
res1 = eval_model(df_out, FEATURES_GDD_ONLY, "GDD-only")
res2 = eval_model(df_out, FEATURES_FULL, "GDD + phenology(+suitability)")

summary = pd.concat([res1, res2], ignore_index=True)
summary = summary.sort_values(["Model", "ROC_AUC"], ascending=[True, False])

print("\n=== COMPARISON SUMMARY ===")
print(summary[["FeatureSet","Model","ROC_AUC","PR_AUC","train_n","test_n","y_train_mean","y_test_mean"]]
      .to_string(index=False))



### 첫 발생 관측 이전 GR값 합산

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

# =========================
# 0) 경로
# =========================
LONG2_PATH = Path("/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2.csv")
FUZZY_ROOT = Path("/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/5Fuzzy")
OUT_PATH   = Path("/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv")

# =========================
# 1) 컬럼 설정
# =========================
SITE_COL   = "site_id"
YEAR_COL   = "year"
DORM_COL   = "dormancy_end_date"   # DOY (정수)
OBS_DOY_COL = "obs_doy"            # 관측일 DOY

# fuzzy 기본 컬럼명
FZ_START  = "start_date"   # YYYY-MM-DD
FZ_GR     = "GR"
FZ_OFFSET = "offset_days"
FZ_WINDOW = "window_idx"

# =========================
# 2) 유틸
# =========================
def read_csv_flexible(p: Path) -> pd.DataFrame:
    for enc in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
        try:
            return pd.read_csv(p, encoding=enc, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(p, low_memory=False)

def pick_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    norm_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in norm_map:
            return norm_map[key]
    return None

def parse_site_from_filename(name: str) -> str | None:
    m = re.search(r"site-([0-9_]+)", name)
    return m.group(1) if m else None

def parse_year_from_filename(name: str) -> int | None:
    m = re.search(r"fuzzy_(\d{4})_site-", name)
    return int(m.group(1)) if m else None

# =========================
# 3) LONG2 로드
# =========================
df = read_csv_flexible(LONG2_PATH)

need = [SITE_COL, YEAR_COL, DORM_COL, OBS_DOY_COL]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"[ERROR] LONG2에 필요한 컬럼이 없습니다: {missing}")

df[SITE_COL] = df[SITE_COL].astype(str).str.strip()
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce").astype("Int64")
df[DORM_COL] = pd.to_numeric(df[DORM_COL], errors="coerce").astype("Int64")
df[OBS_DOY_COL] = pd.to_numeric(df[OBS_DOY_COL], errors="coerce").astype("Int64")

df = df[df[YEAR_COL].notna() & df[DORM_COL].notna() & df[OBS_DOY_COL].notna()].copy()
df[YEAR_COL] = df[YEAR_COL].astype(int)
df[DORM_COL] = df[DORM_COL].astype(int)
df[OBS_DOY_COL] = df[OBS_DOY_COL].astype(int)

print("[INFO] LONG2 valid rows:", len(df))

# =========================
# 4) fuzzy 파일 인덱싱: (site,year) -> filepath
# =========================
fuzzy_files = list(FUZZY_ROOT.rglob("fuzzy_*_site-*.csv"))
index: dict[tuple[str, int], Path] = {}
for p in fuzzy_files:
    y = parse_year_from_filename(p.name)
    s = parse_site_from_filename(p.name)
    if y is None or s is None:
        continue
    index[(str(s), int(y))] = p
print("[INFO] fuzzy files indexed:", len(index))

# =========================
# 5) fuzzy 캐시 (파일 안 바꾸고 메모리에서만 _start_doy 생성)
# =========================
fuzzy_cache: dict[tuple[str, int], pd.DataFrame | None] = {}

def get_fuzzy(site: str, year: int) -> pd.DataFrame | None:
    key = (str(site), int(year))
    if key in fuzzy_cache:
        return fuzzy_cache[key]

    fp = index.get(key)
    if fp is None or (not fp.exists()):
        fuzzy_cache[key] = None
        return None

    fz = read_csv_flexible(fp)

    start_col  = FZ_START  if FZ_START  in fz.columns else pick_col(fz, [FZ_START, "start", "startDate"])
    gr_col     = FZ_GR     if FZ_GR     in fz.columns else pick_col(fz, [FZ_GR, "gr", "GR_value"])
    offset_col = FZ_OFFSET if FZ_OFFSET in fz.columns else pick_col(fz, [FZ_OFFSET, "offset", "offset_day", "offsetDays"])
    window_col = FZ_WINDOW if FZ_WINDOW in fz.columns else pick_col(fz, [FZ_WINDOW, "window", "window_id", "windowIndex"])

    if start_col is None or gr_col is None or offset_col is None or window_col is None:
        print("[WARN] fuzzy 필수 컬럼 없음:", fp.name)
        fuzzy_cache[key] = None
        return None

    # 날짜 → DOY 변환 (파일에는 영향 X, 메모리에서만)
    fz[start_col]  = pd.to_datetime(fz[start_col], errors="coerce").dt.normalize()
    fz[gr_col]     = pd.to_numeric(fz[gr_col], errors="coerce")
    fz[offset_col] = pd.to_numeric(fz[offset_col], errors="coerce")
    fz[window_col] = pd.to_numeric(fz[window_col], errors="coerce")

    fz = fz[fz[start_col].notna() & fz[gr_col].notna()].copy()
    if fz.empty:
        fuzzy_cache[key] = None
        return None

    fz["_start_date"] = fz[start_col]
    fz["_start_doy"]  = fz["_start_date"].dt.dayofyear
    fz["_GR"]         = fz[gr_col]
    fz["_offset"]     = fz[offset_col]
    fz["_window"]     = fz[window_col]

    fz = fz.sort_values("_start_doy").reset_index(drop=True)
    fuzzy_cache[key] = fz
    return fz

# =========================
# 6) row-wise GR feature 생성
#    - dormancy_end_doy 이후 첫 window에서 offset 선택
#    - 선택된 offset의 window들을 obs_doy까지 누적
# =========================
site_idx = df.columns.get_loc(SITE_COL)
year_idx = df.columns.get_loc(YEAR_COL)
dorm_idx = df.columns.get_loc(DORM_COL)
obsd_idx = df.columns.get_loc(OBS_DOY_COL)

out = {
    "GR_sum_until_obs": [],
    "GR_mean_until_obs": [],
    "GR_terms_until_obs": [],
    "chosen_offset": [],
    "chosen_window": [],
    "chosen_start_doy": [],
}

N = len(df)
for i in range(N):
    site    = str(df.iat[i, site_idx]).strip()
    year    = int(df.iat[i, year_idx])
    dorm_d  = int(df.iat[i, dorm_idx])      # dormancy_end DOY
    obs_doy = int(df.iat[i, obsd_idx])      # observation DOY

    fz = get_fuzzy(site, year)
    if fz is None or fz.empty:
        out["GR_sum_until_obs"].append(np.nan)
        out["GR_mean_until_obs"].append(np.nan)
        out["GR_terms_until_obs"].append(np.nan)
        out["chosen_offset"].append(np.nan)
        out["chosen_window"].append(np.nan)
        out["chosen_start_doy"].append(np.nan)
        continue

    # 1) dormancy_end 이후 window들 중에서
    cand = fz[fz["_start_doy"] >= dorm_d]
    if cand.empty:
        # 이 케이스는 드물겠지만 안전장치
        out["GR_sum_until_obs"].append(0.0)
        out["GR_mean_until_obs"].append(0.0)
        out["GR_terms_until_obs"].append(0)
        out["chosen_offset"].append(np.nan)
        out["chosen_window"].append(np.nan)
        out["chosen_start_doy"].append(np.nan)
        continue

    # 2) dormancy_end 이후 가장 이른 window 선택 → offset 결정
    chosen = cand.iloc[0]
    chosen_offset = chosen["_offset"]
    chosen_window = chosen["_window"]
    chosen_start_doy = int(chosen["_start_doy"])

    # 3) 그 offset의 window들 중 [chosen_start_doy ~ obs_doy] 구간의 GR 누적
    sel = fz[
        (fz["_offset"] == chosen_offset) &
        (fz["_start_doy"] >= chosen_start_doy) &
        (fz["_start_doy"] <= obs_doy)
    ]

    if sel.empty:
        out["GR_sum_until_obs"].append(0.0)
        out["GR_mean_until_obs"].append(0.0)
        out["GR_terms_until_obs"].append(0)
    else:
        gr_vals = sel["_GR"].to_numpy(dtype=float)
        out["GR_sum_until_obs"].append(float(gr_vals.sum()))
        out["GR_mean_until_obs"].append(float(gr_vals.mean()))
        out["GR_terms_until_obs"].append(int(len(gr_vals)))

    out["chosen_offset"].append(int(chosen_offset))
    out["chosen_window"].append(int(chosen_window))
    out["chosen_start_doy"].append(chosen_start_doy)

    if (i + 1) % 20000 == 0:
        print(f"[PROGRESS] {i+1:,}/{N:,}")

gr_feat = pd.DataFrame(out)
print("[INFO] GR features rows:", len(gr_feat))

# =========================
# 7) df에 붙이고 타입 정리
# =========================
df2 = df.reset_index(drop=True)

# 혹시 같은 이름 컬럼이 이미 있다면 제거
drop_cols = list(out.keys())
df2 = df2.drop(columns=[c for c in drop_cols if c in df2.columns], errors="ignore")

df_out = pd.concat([df2, gr_feat], axis=1)

# GR_sum/mean은 float로 유지
for c in ["GR_sum_until_obs", "GR_mean_until_obs"]:
    df_out[c] = pd.to_numeric(df_out[c], errors="coerce").astype(float)

# 나머지는 Int64 (NaN 허용)
for c in ["GR_terms_until_obs", "chosen_offset", "chosen_window", "chosen_start_doy"]:
    df_out[c] = pd.to_numeric(df_out[c], errors="coerce").round(0).astype("Int64")

# =========================
# 8) 저장
# =========================
df_out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("[SAVED]", OUT_PATH)

print("NA ratio (chosen_offset):", df_out["chosen_offset"].isna().mean())
print("NA ratio (chosen_window):", df_out["chosen_window"].isna().mean())
print(df_out[[
    SITE_COL, YEAR_COL, OBS_DOY_COL,
    "chosen_offset", "chosen_window",
    "GR_sum_until_obs", "GR_mean_until_obs"
]].head())


### GDD-only / GR-only / GDD+GR

In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# =========================
# 0) 데이터 로드
# =========================
DATA_PATH = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv"

LABEL_COL = "label_event"
YEAR_COL  = "year"

GDD_COL = "GDD10_since_db"
GR_COL  = "GR_sum_until_obs"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", low_memory=False)

# 필수 컬럼 체크
need = [LABEL_COL, YEAR_COL, GDD_COL, GR_COL]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"[ERROR] 필요한 컬럼 없음: {missing}")

# label 정리
df = df[df[LABEL_COL].notna()].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# =========================
# 1) 연도 기준 train / test split
# =========================
years = np.sort(df[YEAR_COL].unique())
n_test_years = max(1, int(np.ceil(len(years) * 0.2)))

test_years  = years[-n_test_years:]
train_years = years[:-n_test_years]

df_train = df[df[YEAR_COL].isin(train_years)].copy()
df_test  = df[df[YEAR_COL].isin(test_years)].copy()

print("train years:", train_years)
print("test years :", test_years)
print("train size :", len(df_train))
print("test size  :", len(df_test))

# =========================
# 2) 모델 함수
# =========================
def run_lr_auc(df_tr, df_te, feature_cols):
    """
    Logistic Regression + StandardScaler
    """
    X_tr = df_tr[feature_cols]
    y_tr = df_tr[LABEL_COL]

    X_te = df_te[feature_cols]
    y_te = df_te[LABEL_COL]

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

    pipe.fit(X_tr, y_tr)

    proba = pipe.predict_proba(X_te)[:, 1]

    roc = roc_auc_score(y_te, proba)
    pr  = average_precision_score(y_te, proba)

    return roc, pr

# =========================
# 3) 실험 3종
# =========================
results = []

# (1) GDD-only
roc, pr = run_lr_auc(df_train, df_test, [GDD_COL])
results.append({
    "Model": "GDD-only",
    "Features": GDD_COL,
    "ROC-AUC": roc,
    "PR-AUC": pr
})

# (2) GR-only
roc, pr = run_lr_auc(df_train, df_test, [GR_COL])
results.append({
    "Model": "GR-only",
    "Features": GR_COL,
    "ROC-AUC": roc,
    "PR-AUC": pr
})

# (3) GDD + GR
roc, pr = run_lr_auc(df_train, df_test, [GDD_COL, GR_COL])
results.append({
    "Model": "GDD + GR",
    "Features": f"{GDD_COL} + {GR_COL}",
    "ROC-AUC": roc,
    "PR-AUC": pr
})

# =========================
# 4) 결과 테이블
# =========================
res_df = pd.DataFrame(results)
print("\n=== Logistic Regression AUC Comparison ===")
print(res_df.round(4))


In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# =========================
# 0) 데이터 로드
# =========================
DATA_PATH = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv"

LABEL_COL = "label_event"
YEAR_COL  = "year"

GDD_COL = "GDD10_since_db"
GR_COL  = "GR_sum_until_obs"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", low_memory=False)

# 필수 컬럼 체크
need = [LABEL_COL, YEAR_COL, GDD_COL, GR_COL]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"[ERROR] 필요한 컬럼 없음: {missing}")

# label 정리
df = df[df[LABEL_COL].notna()].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# =========================
# 1) 연도 기준 train / test split
# =========================
years = np.sort(df[YEAR_COL].unique())
n_test_years = max(1, int(np.ceil(len(years) * 0.2)))

test_years  = years[-n_test_years:]
train_years = years[:-n_test_years]

df_train = df[df[YEAR_COL].isin(train_years)].copy()
df_test  = df[df[YEAR_COL].isin(test_years)].copy()

print("train years:", train_years)
print("test years :", test_years)
print("train size :", len(df_train))
print("test size  :", len(df_test))

print("[INFO] train positive rate:", df_train[LABEL_COL].mean())
print("[INFO] test  positive rate:", df_test[LABEL_COL].mean())

# =========================
# 2) RF 모델 함수
# =========================
def run_rf_auc(df_tr, df_te, feature_cols):
    """
    RandomForestClassifier + SimpleImputer
    (스케일링은 필요 없음)
    """
    X_tr = df_tr[feature_cols]
    y_tr = df_tr[LABEL_COL]

    X_te = df_te[feature_cols]
    y_te = df_te[LABEL_COL]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=42,
        )),
    ])

    pipe.fit(X_tr, y_tr)

    proba = pipe.predict_proba(X_te)[:, 1]

    roc = roc_auc_score(y_te, proba)
    pr  = average_precision_score(y_te, proba)

    return roc, pr

# =========================
# 3) 실험 3종 (RF)
# =========================
results = []

# (1) GDD-only
roc, pr = run_rf_auc(df_train, df_test, [GDD_COL])
results.append({
    "Model": "GDD-only (RF)",
    "Features": GDD_COL,
    "ROC-AUC": roc,
    "PR-AUC": pr
})

# (2) GR-only
roc, pr = run_rf_auc(df_train, df_test, [GR_COL])
results.append({
    "Model": "GR-only (RF)",
    "Features": GR_COL,
    "ROC-AUC": roc,
    "PR-AUC": pr
})

# (3) GDD + GR
roc, pr = run_rf_auc(df_train, df_test, [GDD_COL, GR_COL])
results.append({
    "Model": "GDD + GR (RF)",
    "Features": f"{GDD_COL} + {GR_COL}",
    "ROC-AUC": roc,
    "PR-AUC": pr
})

# =========================
# 4) 결과 테이블
# =========================
res_df = pd.DataFrame(results)
print("\n=== RandomForest AUC Comparison ===")
print(res_df.round(4))


In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score

# XGBoost / LightGBM
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# =========================
# 0) 데이터 로드
# =========================
DATA_PATH = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv"

LABEL_COL = "label_event"
YEAR_COL  = "year"

GDD_COL = "GDD10_since_db"
GR_COL  = "GR_sum_until_obs"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", low_memory=False)

# 필수 컬럼 체크
need = [LABEL_COL, YEAR_COL, GDD_COL, GR_COL]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"[ERROR] 필요한 컬럼 없음: {missing}")

# label 정리
df = df[df[LABEL_COL].notna()].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# =========================
# 1) 연도 기준 train / test split
# =========================
years = np.sort(df[YEAR_COL].unique())
n_test_years = max(1, int(np.ceil(len(years) * 0.2)))

test_years  = years[-n_test_years:]
train_years = years[:-n_test_years]

df_train = df[df[YEAR_COL].isin(train_years)].copy()
df_test  = df[df[YEAR_COL].isin(test_years)].copy()

print("train years:", train_years)
print("test years :", test_years)
print("train size :", len(df_train))
print("test size  :", len(df_test))
print("[INFO] train positive rate:", df_train[LABEL_COL].mean())
print("[INFO] test  positive rate:", df_test[LABEL_COL].mean())

# =========================
# 2) 공통 평가 함수
# =========================
def eval_model(model, df_tr, df_te, feature_cols):
    X_tr = df_tr[feature_cols]
    y_tr = df_tr[LABEL_COL]

    X_te = df_te[feature_cols]
    y_te = df_te[LABEL_COL]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])

    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1]

    roc = roc_auc_score(y_te, proba)
    pr  = average_precision_score(y_te, proba)
    return roc, pr

# =========================
# 3) XGBoost / LightGBM 설정
# =========================
def make_xgb():
    # 파라미터는 약간 보수적으로 설정 (너 테스트해보면서 튜닝해도 됨)
    return XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",   # Mac ARM이면 "hist"/"approx" 추천
        n_jobs=-1,
        random_state=42,
    )

def make_lgb():
    return LGBMClassifier(
        n_estimators=400,
        max_depth=-1,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary",
        n_jobs=-1,
        random_state=42,
    )

# =========================
# 4) 실험 3종 × 2모델
# =========================
results = []

feature_sets = [
    ("GDD-only",           [GDD_COL]),
    ("GR-only",            [GR_COL]),
    ("GDD + GR",           [GDD_COL, GR_COL]),
]

# ---- XGBoost ----
for name, feats in feature_sets:
    roc, pr = eval_model(make_xgb(), df_train, df_test, feats)
    results.append({
        "Algo": "XGBoost",
        "FeatureSet": name,
        "Features": " + ".join(feats),
        "ROC-AUC": roc,
        "PR-AUC": pr,
    })

# ---- LightGBM ----
for name, feats in feature_sets:
    roc, pr = eval_model(make_lgb(), df_train, df_test, feats)
    results.append({
        "Algo": "LightGBM",
        "FeatureSet": name,
        "Features": " + ".join(feats),
        "ROC-AUC": roc,
        "PR-AUC": pr,
    })

# =========================
# 5) 결과 테이블
# =========================
res_df = pd.DataFrame(results)
print("\n=== XGBoost / LightGBM AUC Comparison ===")
print(res_df.sort_values(["Algo", "FeatureSet"]).round(4))


In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

from xgboost import XGBClassifier

# =========================
# 0) 데이터 로드
# =========================
DATA_PATH = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv"

LABEL_COL = "label_event"
YEAR_COL  = "year"

GDD_COL = "GDD10_since_db"
GR_COL  = "GR_sum_until_obs"

# phenology / DOY 관련 후보 컬럼들
PHENO_CANDIDATES = [
    "days_since_flowering",
    "days_since_growing_start",
    "days_until_growing_end",
    "is_growing",
    "is_dormancy",
    "obs_doy",       # 있으면 사용
]

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", low_memory=False)

# 필수 컬럼 체크
need = [LABEL_COL, YEAR_COL, GDD_COL, GR_COL]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"[ERROR] 필요한 컬럼 없음: {missing}")

# 실제 존재하는 phenology 컬럼만 사용
PHENO_COLS = [c for c in PHENO_CANDIDATES if c in df.columns]
print("[INFO] 사용 가능한 phenology cols:", PHENO_COLS)

# label 정리
df = df[df[LABEL_COL].notna()].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# =========================
# 1) 연도 기준 train / test split
# =========================
years = np.sort(df[YEAR_COL].unique())
n_test_years = max(1, int(np.ceil(len(years) * 0.2)))

test_years  = years[-n_test_years:]
train_years = years[:-n_test_years]

df_train = df[df[YEAR_COL].isin(train_years)].copy()
df_test  = df[df[YEAR_COL].isin(test_years)].copy()

print("train years:", train_years)
print("test years :", test_years)
print("train size :", len(df_train))
print("test size  :", len(df_test))
print("[INFO] train positive rate:", df_train[LABEL_COL].mean())
print("[INFO] test  positive rate:", df_test[LABEL_COL].mean())

# =========================
# 2) 모델 팩토리 & 공통 평가 함수
# =========================
def make_lr():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

def make_rf():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=42,
        ))
    ])

def make_xgb():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("xgb", XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
        ))
    ])

def eval_model(pipe, df_tr, df_te, feature_cols):
    X_tr = df_tr[feature_cols]
    y_tr = df_tr[LABEL_COL]

    X_te = df_te[feature_cols]
    y_te = df_te[LABEL_COL]

    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1]

    roc = roc_auc_score(y_te, proba)
    pr  = average_precision_score(y_te, proba)
    return roc, pr

# =========================
# 3) Feature set 정의
# =========================
feature_sets = []

# (A) GDD + GR only (기존 baseline)
feature_sets.append(("GDD+GR_only", [GDD_COL, GR_COL]))

# (B) phenology only
if PHENO_COLS:
    feature_sets.append(("Phenology_only", PHENO_COLS))

# (C) GDD+GR + phenology (핵심)
if PHENO_COLS:
    feature_sets.append(("GDD+GR+Phenology", [GDD_COL, GR_COL] + PHENO_COLS))

print("\n[INFO] 실험할 feature sets:")
for name, cols in feature_sets:
    print(f"  - {name}: {cols}")

# =========================
# 4) LR / RF / XGB × feature set 실험
# =========================
results = []

algo_factories = [
    ("LogisticRegression", make_lr),
    ("RandomForest",       make_rf),
    ("XGBoost",            make_xgb),
]

for algo_name, factory in algo_factories:
    for fs_name, cols in feature_sets:
        roc, pr = eval_model(factory(), df_train, df_test, cols)
        results.append({
            "Algo": algo_name,
            "FeatureSet": fs_name,
            "n_features": len(cols),
            "ROC_AUC": roc,
            "PR_AUC": pr,
        })

# =========================
# 5) 결과 출력
# =========================
res_df = pd.DataFrame(results)
print("\n=== LR / RF / XGB with Phenology Features ===")
print(res_df.sort_values(["Algo", "FeatureSet"]).round(4))


In [ ]:
import pandas as pd
import numpy as np

# =========================
# 설정
# =========================
DATA_PATH = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv"

SITE_COL  = "site_id"
YEAR_COL  = "year"
DATE_COL  = "조사일자(YYYYMMDD)"
LABEL_COL = "label_event"

GDD_COL = "GDD10_since_db"
GR_COL  = "GR_sum_until_obs"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", low_memory=False)

# 타입 정리
df[SITE_COL] = df[SITE_COL].astype(str).str.strip()
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce").astype("Int64")
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce").dt.normalize()
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")

df = df[df[YEAR_COL].notna() & df[DATE_COL].notna() & df[LABEL_COL].notna()].copy()
df[YEAR_COL] = df[YEAR_COL].astype(int)
df[LABEL_COL] = df[LABEL_COL].astype(int)

df[GDD_COL] = pd.to_numeric(df[GDD_COL], errors="coerce")
df[GR_COL]  = pd.to_numeric(df[GR_COL], errors="coerce")

# =========================
# 1) (site,year)별 first_event row 추출
# =========================
pos = df[df[LABEL_COL] == 1].copy()
pos = pos.sort_values([SITE_COL, YEAR_COL, DATE_COL])

first_evt = pos.groupby([SITE_COL, YEAR_COL], as_index=False).head(1).copy()

print("[INFO] total rows:", len(df))
print("[INFO] positive rows:", len(pos))
print("[INFO] first_event pairs:", len(first_evt))

# =========================
# 2) first_event 시점 GDD/GR 결측 체크
# =========================
print("\n[NA ratio at first_event]")
print("GDD NA:", first_evt[GDD_COL].isna().mean())
print("GR  NA:", first_evt[GR_COL].isna().mean())

# 결측 있는 케이스 몇 개 보기
na_cases = first_evt[first_evt[GDD_COL].isna() | first_evt[GR_COL].isna()][
    [SITE_COL, YEAR_COL, DATE_COL, GDD_COL, GR_COL]
].head(20)
print("\n[NA cases sample]")
print(na_cases)

# =========================
# 3) 분포 요약 통계
# =========================
def summarize(series: pd.Series, name: str):
    s = series.dropna()
    if s.empty:
        print(f"\n{name}: (no data)")
        return
    qs = s.quantile([0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
    print(f"\n=== {name} @ first_event ===")
    print("N:", len(s))
    print("min/mean/std/max:", float(s.min()), float(s.mean()), float(s.std()), float(s.max()))
    print("quantiles:\n", qs)

summarize(first_evt[GDD_COL], "GDD10_since_db")
summarize(first_evt[GR_COL],  "GR_sum_until_obs")

# =========================
# 4) 임계값 후보(quantile 기반) 테이블
#    - "first_event의 x%가 이 값 이하/이상" 이런 식으로 해석 가능
# =========================
q_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
thr = pd.DataFrame({
    "quantile": q_list,
    "GDD_thr": [first_evt[GDD_COL].quantile(q) for q in q_list],
    "GR_thr":  [first_evt[GR_COL].quantile(q) for q in q_list],
}).round(2)

print("\n=== Threshold candidates (quantiles of first_event) ===")
print(thr)

# =========================
# 5) (옵션) first_event 결과 저장 (보고서/그래프용)
# =========================
OUT_FIRST = "/Users/doyoung-gil/연구실/데이터/사과/first_event_GDD_GR_table.csv"
first_evt[[SITE_COL, YEAR_COL, DATE_COL, GDD_COL, GR_COL]].to_csv(OUT_FIRST, index=False, encoding="utf-8-sig")
print("\n[SAVED]", OUT_FIRST)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# =========================
# 설정
# =========================
DATA_PATH = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv"

LABEL_COL = "label_event"
GR_COL    = "GR_sum_until_obs"
YEAR_COL  = "year"

THRESHOLDS = [2.4, 3.0, 3.5, 4.0]

# =========================
# 데이터 로드
# =========================
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", low_memory=False)

df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df[GR_COL]    = pd.to_numeric(df[GR_COL], errors="coerce")
df[YEAR_COL]  = pd.to_numeric(df[YEAR_COL], errors="coerce")

df = df[df[LABEL_COL].notna() & df[GR_COL].notna()].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

print("rows used:", len(df))
print("positive rate:", df[LABEL_COL].mean())

# =========================
# 연도 기준 test split (Step 1과 동일)
# =========================
years = np.sort(df[YEAR_COL].dropna().unique())
n_test_years = max(1, int(np.ceil(len(years) * 0.2)))

test_years  = years[-n_test_years:]
train_years = years[:-n_test_years]

df_test = df[df[YEAR_COL].isin(test_years)].copy()

print("test years:", test_years)
print("test size :", len(df_test))
print("test positive rate:", df_test[LABEL_COL].mean())

# =========================
# threshold 평가 함수
# =========================
def eval_threshold(df_eval, thr):
    y_true = df_eval[LABEL_COL].values
    score  = df_eval[GR_COL].values          # 연속 점수
    y_pred = (score >= thr).astype(int)      # rule classifier

    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    roc = roc_auc_score(y_true, score)
    pr  = average_precision_score(y_true, score)

    return prec, rec, f1, roc, pr

# =========================
# 평가 실행
# =========================
rows = []
for thr in THRESHOLDS:
    prec, rec, f1, roc, pr = eval_threshold(df_test, thr)
    rows.append({
        "GR_threshold": thr,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        # "ROC-AUC": roc,
        # "PR-AUC": pr
    })

res = pd.DataFrame(rows).round(4)

print("\n=== GR threshold rule performance (test years) ===")
print(res)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================
# 설정
# =========================
DATA_PATH = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv"

SITE_COL  = "site_id"
YEAR_COL  = "year"
DATE_COL  = "조사일자(YYYYMMDD)"
LABEL_COL = "label_event"

GR_COL = "GR_sum_until_obs"

THRESHOLDS = [2.4, 3.0, 3.5]
K_PRE_LIST = [1, 2, 3]   # first_event 직전 k개 관측

# =========================
# 로드
# =========================
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", low_memory=False)

for c in [SITE_COL, YEAR_COL]:
    df[c] = df[c].astype(str)

df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce").astype(int)
df[GR_COL] = pd.to_numeric(df[GR_COL], errors="coerce")

df = df.dropna(subset=[YEAR_COL, DATE_COL, GR_COL]).copy()
df[YEAR_COL] = df[YEAR_COL].astype(int)

# =========================
# first_event 추출
# =========================
pos = df[df[LABEL_COL] == 1].copy()

first_event = (
    pos.groupby([SITE_COL, YEAR_COL])[DATE_COL]
       .min()
       .reset_index()
       .rename(columns={DATE_COL: "first_event_date"})
)

df = df.merge(first_event, on=[SITE_COL, YEAR_COL], how="inner")

# =========================
# first_event 이전 관측만 사용
# =========================
df_pre = df[df[DATE_COL] < df["first_event_date"]].copy()

# 정렬
df_pre = df_pre.sort_values([SITE_COL, YEAR_COL, DATE_COL])

# =========================
# 평가
# =========================
rows = []

for k in K_PRE_LIST:
    eval_parts = []

    for (site, year), g in df_pre.groupby([SITE_COL, YEAR_COL]):
        if len(g) < k:
            continue

        g = g.sort_values(DATE_COL)

        # 직전 k개 = positive
        g_tail = g.tail(k).copy()
        g_tail["y_true"] = 1

        # 그 이전 = negative
        g_head = g.iloc[:-k].copy()
        g_head["y_true"] = 0

        eval_parts.append(pd.concat([g_head, g_tail], axis=0))

    if not eval_parts:
        continue

    df_eval = pd.concat(eval_parts, ignore_index=True)

    for thr in THRESHOLDS:
        y_true = df_eval["y_true"].values
        y_pred = (df_eval[GR_COL].values >= thr).astype(int)

        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)

        rows.append({
            "k_pre_obs": k,
            "GR_threshold": thr,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "N_samples": len(df_eval)
        })

res = pd.DataFrame(rows).round(4)

print("\n=== Step 2.6: first_event 직전 조기경보 성능 ===")
print(res)


## Survival analysis

In [ ]:
import pandas as pd

# nrows=0을 설정하면 데이터는 안 읽고 컬럼명만 가져옵니다.
df = pd.read_csv('/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv', nrows=0)
column_names = df.columns.tolist()

print(column_names)

#### site-year 단위로 first detection 요약

In [ ]:
import pandas as pd

df = pd.read_csv("/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv")

SITE_COL = "site_id"
YEAR_COL = "year"
DOY_COL = "obs_doy"
LABEL_COL = "label_event"

rows = []
for (site, year), sub in df.groupby([SITE_COL, YEAR_COL]):
    sub = sub.sort_values(DOY_COL)

    detected_days = sub.loc[sub[LABEL_COL] == 1, DOY_COL].values
    if len(detected_days) > 0:
        time = int(detected_days[0])   # 첫 detection day
        event = 1
    else:
        time = int(sub[DOY_COL].max()) # 마지막 관측일까지 안나옴 → censoring
        event = 0

    rows.append({
        SITE_COL: site,
        YEAR_COL: year,
        "time": time,
        "event": event
    })

df_surv_basic = pd.DataFrame(rows)
print(df_surv_basic.head())


#### site-year summary feature 만들기

In [ ]:
df_feat_list = []
for (site, year), sub in df.groupby(["site_id", "year"]):
    sub = sub.sort_values("obs_doy")

    detected = sub[sub["label_event"] == 1]
    if len(detected) > 0:
        row = detected.iloc[0]   # 첫 detection row
    else:
        row = sub.iloc[-1]       # 마지막 관측 row (censored)

    df_feat_list.append({
        "site_id": site,
        "year": year,

        # === GR ===
        "GR_sum_event": row["GR_sum_until_obs"],

        # === GDD ===
        # (이 값이 누적인지 daily인지 나중에 확인할거임)
        "GDD_event": row["GDD10_since_db"] if "GDD10_since_db" in row else None,

        # === suitability ===
        # (suitability가 site-year 고정이면 그대로 잘 들어갈거임)
        "suitability_event": row["best_suitability"] if "best_suitability" in row else None,
    })

df_feat = pd.DataFrame(df_feat_list)
print(df_feat.head())


#### Feature merge

In [ ]:
df_surv = df_surv_basic.merge(
    df_feat,
    on=["site_id", "year"],
    how="left"
)
print(df_surv.head())

#### Cox 모델 돌리기

In [ ]:
from lifelines import CoxPHFitter

# 1) 사용할 feature들 지정
feature_cols = ["GR_sum_event", "GDD_event", "suitability_event"]

# 2) 결측 제거 (중요)
df_model = df_surv.dropna(subset=["time", "event"] + feature_cols)

# 3) Cox 모델 생성 & 학습
cph = CoxPHFitter()
cph.fit(
    df_model[["time", "event"] + feature_cols],
    duration_col="time",
    event_col="event"
)

# 4) 결과 출력
cph.print_summary()


In [ ]:
import seaborn as sns
sns.boxplot(data=df_surv, x='event', y='GR_sum_event')


In [ ]:
df_event = df[df["label_event"] == 1].copy()

# first detection per site-year (혹시 여러 번 잡힌 날 있을 수 있으므로 min)
df_event_first = (
    df_event.groupby(["site_id", "year"])["obs_doy"]
             .min()
             .reset_index()
             .rename(columns={"obs_doy": "first_detection_doy"})
)


In [ ]:
df_DOY = df_event_first.merge(df_feat, on=["site_id", "year"], how="left")
print(df_DOY.head())

In [ ]:
df_DOY[["first_detection_doy", "GDD_event", "GR_sum_event", "suitability_event"]].corr()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# --- GDD vs DOY ---
sns.scatterplot(
    data=df_DOY,
    x="GDD_event",
    y="first_detection_doy",
    ax=ax[0]
)
ax[0].set_title("GDD_event vs First Detection DOY")
ax[0].set_xlabel("GDD_event")
ax[0].set_ylabel("first_detection_doy")

# --- GR vs DOY ---
sns.scatterplot(
    data=df_DOY,
    x="GR_sum_event",
    y="first_detection_doy",
    ax=ax[1]
)
ax[1].set_title("GR_sum_event vs First Detection DOY")
ax[1].set_xlabel("GR_sum_event")
ax[1].set_ylabel("first_detection_doy")

plt.tight_layout()
plt.show()


In [ ]:
import statsmodels.api as sm

X = df_DOY[["GDD_event", "GR_sum_event"]]
X = sm.add_constant(X)
y = df_DOY["first_detection_doy"]

model = sm.OLS(y, X).fit()
print(model.summary())


### time-varying

In [ ]:
import pandas as pd

df = pd.read_csv("/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2_with_GR.csv")


In [ ]:
import pandas as pd

df_obs = df.copy()


In [ ]:
rows = []

for (site, year), sub in df_obs.groupby(["site_id", "year"]):
    sub = sub.sort_values("obs_doy").reset_index(drop=True)

    # 첫 detection day 찾기
    detection_days = sub.loc[sub["label_event"] == 1, "obs_doy"]
    if len(detection_days) > 0:
        first_event_day = detection_days.iloc[0]
    else:
        first_event_day = None

    # interval 생성
    for i in range(len(sub)):
        if i == 0:
            start = sub.loc[i, "obs_doy"]
        else:
            start = sub.loc[i-1, "obs_doy"]

        stop = sub.loc[i, "obs_doy"]

        # event marking (first event day에서만 event=1)
        if first_event_day is not None and stop == first_event_day:
            event = 1
        else:
            event = 0

        rows.append({
            "site_id": site,
            "year": year,
            "start": start,
            "stop": stop,
            "event": event,
            "GDD": sub.loc[i, "GDD10_since_db"],
            "GR": sub.loc[i, "GR_sum_until_obs"],
        })

df_tv = pd.DataFrame(rows)


In [ ]:
df_tv["id"] = df_tv["site_id"].astype(str) + "_" + df_tv["year"].astype(str)


In [ ]:
print(df_tv.head())
print(df_tv.tail())

print(df_tv["event"].value_counts())
print(df_tv.groupby("id")["event"].sum().value_counts())


In [ ]:
import numpy as np
import pandas as pd

# 0) 정렬
df_tv = df_tv.sort_values(["id", "stop", "start"]).reset_index(drop=True)

# 1) 원래 event 백업
df_tv["event_raw"] = df_tv["event"].astype(int)

# 2) 같은 id에서 같은 stop(같은 조사일)이 여러 번 있으면 1개로 collapse
#    - start는 가장 작은 값(앞 interval)로
#    - covariate(GDD, GR)는 그날의 값(last)로
#    - event_raw는 max로 (그날에 event가 있으면 1)
df_tv = (df_tv
    .groupby(["id", "stop"], as_index=False)
    .agg({
        "site_id": "first",
        "year": "first",
        "start": "min",
        "GDD": "last",
        "GR": "last",
        "event_raw": "max",
    })
)

# 3) id당 event는 딱 1번만: 가장 이른 stop에서만 event=1
df_tv["event"] = 0
first_event_idx = (df_tv[df_tv["event_raw"] == 1]
                   .sort_values(["id", "stop"])
                   .groupby("id")
                   .head(1)
                   .index)
df_tv.loc[first_event_idx, "event"] = 1

df_tv = df_tv.drop(columns=["event_raw"])

# 4) start==stop(0길이) 처리: 있으면 start를 stop-1로 보정
mask = df_tv["stop"] <= df_tv["start"]
df_tv.loc[mask, "start"] = df_tv.loc[mask, "stop"] - 1

# 5) 체크
print("event per id:")
print(df_tv.groupby("id")["event"].sum().value_counts().sort_index())
print("non-positive intervals:", (df_tv["stop"] <= df_tv["start"]).sum())
print(df_tv.head())


#### GDD 단독

In [ ]:
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(df_tv, id_col="id", start_col="start", stop_col="stop", event_col="event",
        show_progress=True, formula="GDD")
ctv.print_summary()


#### GR 단독 

In [ ]:
ctv = CoxTimeVaryingFitter()
ctv.fit(df_tv, id_col="id", start_col="start", stop_col="stop", event_col="event",
        show_progress=True, formula="GR")
ctv.print_summary()


#### GDD + GR

In [ ]:
ctv = CoxTimeVaryingFitter()
ctv.fit(df_tv, id_col="id", start_col="start", stop_col="stop", event_col="event",
        show_progress=True, formula="GDD + GR")
ctv.print_summary()


## Transformer

### 데이터 전처리

In [ ]:
import pandas as pd
import numpy as np

# 기상데이터 + 관측데이터 + GDD join 파일
PATH_DAILY = "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/joined_SAMPLE_GDD_2013_2024_APPLE_복숭아순나방.csv"

# 관측(불규칙) + GR/fuzzy 피처 파일
PATH_OBS   = "/Users/doyoung-gil/연구실/데이터/사과/peach_moth_obs_with_fuzzy_features_LONG2.csv"

# threshold (일단 3으로 시작)
THRESHOLD = 1


In [ ]:
# daily(joined) 읽기 + 표준 키 만들기
usecols_daily = [
    "지점ID", "일시",
    "일강수량(mm)", "최고기온(°C)", "최저기온(°C)", "평균기온(°C)",
    "평균 풍속(m/s)", "최대 풍속(m/s)",
    "DD10", "GDD10_cum",
    "(트랩)복숭아순나방-마리수",  # 있어도 되고 없어도 됨
]

daily = pd.read_csv(PATH_DAILY, usecols=usecols_daily)

daily["date"] = pd.to_datetime(daily["일시"], errors="coerce")
daily = daily.dropna(subset=["date"]).copy()

daily = daily.rename(columns={"지점ID": "site_id"})
daily["year"] = daily["date"].dt.year
daily["doy"]  = daily["date"].dt.dayofyear
daily = daily.sort_values(["site_id", "date"]).reset_index(drop=True)

daily.head()


In [ ]:
# “일별 연속성” 체크 (site-year별 1~365가 있는지)
def check_daily_grid(daily_df, site, year):
    sub = daily_df[(daily_df["site_id"] == site) & (daily_df["year"] == year)]
    days = set(sub["doy"].astype(int).tolist())
    missing = [d for d in range(1, 366) if d not in days]
    return len(sub), len(missing), missing[:10]

tmp_key = daily[["site_id","year"]].drop_duplicates().iloc[0]
nrows, nmiss, miss10 = check_daily_grid(daily, tmp_key["site_id"], tmp_key["year"])
print("example:", tmp_key.to_dict(), "rows:", nrows, "missing:", nmiss, "miss10:", miss10)


In [ ]:
# obs(LONG2) 읽기 + 관측일 정리
obs = pd.read_csv(PATH_OBS)

# 필수 컬럼 체크
need = ["site_id", "year", "obs_doy", "(트랩)복숭아순나방-마리수"]
missing = [c for c in need if c not in obs.columns]
if missing:
    raise ValueError(f"Missing columns in LONG2: {missing}")

# obs_doy 숫자화
obs["obs_doy"] = pd.to_numeric(obs["obs_doy"], errors="coerce")
obs = obs.dropna(subset=["obs_doy"]).copy()
obs["obs_doy"] = obs["obs_doy"].astype(int)

# 정렬
obs = obs.sort_values(["site_id", "year", "obs_doy"]).reset_index(drop=True)

print("obs shape:", obs.shape)
obs.head()


In [ ]:
print("obs shape:", obs.shape)
print("obs columns:", obs.columns.tolist()[:50])
print("obs_date na:", obs["obs_date"].isna().mean() if "obs_date" in obs.columns else "NO obs_date col")

# group key가 진짜 있는지
print("has site_id:", "site_id" in obs.columns)
print("has year:", "year" in obs.columns)

# 트랩 컬럼 확인(비슷한 이름 찾기)
cand = [c for c in obs.columns if "복숭아" in c or "트랩" in c or "마리" in c]
print("candidate count cols:", cand)


In [ ]:
#(L, R] 라벨 생성 — obs_doy로만 만들기 (보간 X)
# def build_interval_labels_from_doy(
#     obs_df: pd.DataFrame,
#     threshold: float = 3.0,
#     count_col: str = "(트랩)복숭아순나방-마리수",
#     season_start_doy: int = 1,
#     season_end_doy: int = 365,
# ):
#     rows = []
#     for (site, year), sub in obs_df.groupby(["site_id", "year"], sort=False):
#         sub = sub.sort_values("obs_doy")

#         y = sub[count_col].fillna(0).to_numpy()
#         t = sub["obs_doy"].to_numpy().astype(int)

#         above = y >= threshold

#         if above.any():
#             idx_R = int(np.argmax(above))
#             R_doy = int(t[idx_R])

#             if idx_R == 0:
#                 censor_type = "left"
#                 L_doy = season_start_doy
#             else:
#                 idx_Ls = np.where(~above[:idx_R])[0]
#                 if len(idx_Ls) == 0:
#                     censor_type = "left"
#                     L_doy = season_start_doy
#                 else:
#                     censor_type = "interval"
#                     L_doy = int(t[idx_Ls[-1]])

#             rows.append({
#                 "site_id": site,
#                 "year": int(year),
#                 "censor_type": censor_type,
#                 "L_doy": int(L_doy),
#                 "R_doy": int(R_doy),
#                 "threshold": threshold,
#             })

#         else:
#             rows.append({
#                 "site_id": site,
#                 "year": int(year),
#                 "censor_type": "right",
#                 "L_doy": season_start_doy,
#                 "R_doy": season_end_doy,
#                 "threshold": threshold,
#             })

#     labels = pd.DataFrame(rows)
#     if labels.empty:
#         raise ValueError("labels came out empty. Check grouping keys and obs_doy.")
#     return labels


# labels = build_interval_labels_from_doy(obs, threshold=THRESHOLD)
# print(labels["censor_type"].value_counts())
# labels.head()


In [ ]:
# 같은 날 여러 번 관측이 있으면 (트랩)복숭아순나방-마리수의 최대값으로 대표
THRESHOLD = 1
count_col = "(트랩)복숭아순나방-마리수"

# obs는 LONG2에서 읽어온 원본 DataFrame이라고 가정
obs2 = obs.copy()

# obs_doy 정리
obs2["obs_doy"] = pd.to_numeric(obs2["obs_doy"], errors="coerce")
obs2 = obs2.dropna(subset=["obs_doy"]).copy()
obs2["obs_doy"] = obs2["obs_doy"].astype(int)

# ✅ 1일 1레코드: 그날 최대 포획수로 집계
obs2 = (obs2
        .groupby(["site_id", "year", "obs_doy"], as_index=False)[count_col]
        .max()
       )

print("obs2 shape:", obs2.shape)
obs2.head()

In [ ]:
def build_interval_labels_from_doy_fixed(
    obs_df: pd.DataFrame,
    threshold: float = 3.0,
    count_col: str = "(트랩)복숭아순나방-마리수",
    season_start_doy: int = 1,
    season_end_doy: int = 365,
):
    rows = []
    for (site, year), sub in obs_df.groupby(["site_id", "year"], sort=False):
        sub = sub.sort_values("obs_doy")

        y = sub[count_col].fillna(0).to_numpy()
        t = sub["obs_doy"].to_numpy().astype(int)

        above = y >= threshold

        if above.any():
            idx_R = int(np.argmax(above))
            R_doy = int(t[idx_R])

            if idx_R == 0:
                censor_type = "left"
                L_doy = season_start_doy
            else:
                idx_Ls = np.where(~above[:idx_R])[0]
                if len(idx_Ls) == 0:
                    censor_type = "left"
                    L_doy = season_start_doy
                else:
                    censor_type = "interval"
                    L_doy = int(t[idx_Ls[-1]])

            # ✅ 핵심: interval인데 L>=R이면 (R-1, R]로 강제
            if censor_type == "interval" and L_doy >= R_doy:
                L_doy = max(season_start_doy, R_doy - 1)

            rows.append({
                "site_id": site,
                "year": int(year),
                "censor_type": censor_type,
                "L_doy": int(L_doy),
                "R_doy": int(R_doy),
                "threshold": threshold,
            })
        else:
            rows.append({
                "site_id": site,
                "year": int(year),
                "censor_type": "right",
                "L_doy": season_start_doy,
                "R_doy": season_end_doy,
                "threshold": threshold,
            })

    return pd.DataFrame(rows)

labels = build_interval_labels_from_doy_fixed(obs2, threshold=THRESHOLD, count_col=count_col)
print(labels["censor_type"].value_counts())
labels.head()


In [ ]:
# daily(연속 일별) + labels(구간 라벨) merge

# daily는 Step1에서 만든 데이터라고 가정
# daily columns: site_id, date, year, doy, ... weather..., DD10, GDD10_cum

# Transformer 입력으로 쓸 feature들 (원하는 만큼 추가 가능)
feature_cols = [
    "일강수량(mm)",
    "최고기온(°C)",
    "최저기온(°C)",
    "평균기온(°C)",
    "평균 풍속(m/s)",
    "최대 풍속(m/s)",
    "DD10",
    "GDD10_cum",
]

# 필요한 컬럼만
daily_feat = daily[["site_id", "year", "doy", "date"] + feature_cols].copy()

# labels 붙이기 (site-year 기준)
train_df = daily_feat.merge(labels, on=["site_id", "year"], how="inner")

print("train_df shape:", train_df.shape)
train_df.head()


In [ ]:
# 기상데이터 NaN 보간
weather_cols = [
    "일강수량(mm)",
    "최고기온(°C)",
    "최저기온(°C)",
    "평균기온(°C)",
    "평균 풍속(m/s)",
    "최대 풍속(m/s)",
]

def fill_weather_nan_by_group(train_df, weather_cols):
    out = []
    for (site, year), sub in train_df.groupby(["site_id","year"], sort=False):
        sub = sub.sort_values("doy").copy()
        for c in weather_cols:
            sub[c] = pd.to_numeric(sub[c], errors="coerce")

            # 선형 보간 (앞/뒤 모두 허용)
            sub[c] = sub[c].interpolate(limit_direction="both")

            # 혹시 맨 앞/맨 뒤가 비면 ffill/bfill로 마무리
            sub[c] = sub[c].ffill().bfill()

        # (선택) 강수는 남은 NaN이 있으면 0으로
        sub["일강수량(mm)"] = sub["일강수량(mm)"].fillna(0.0)

        out.append(sub)

    return pd.concat(out, ignore_index=True)

train_df2 = fill_weather_nan_by_group(train_df, weather_cols)

# 확인
print("NaN rate after fill:")
print(train_df2[weather_cols].isna().mean())


In [ ]:
# site-year별로 DOY가 연속인지 점검
def missing_doys_one_group(df, site, year):
    sub = df[(df["site_id"] == site) & (df["year"] == year)].sort_values("doy")
    have = set(sub["doy"].astype(int).tolist())
    miss = [d for d in range(1, 366) if d not in have]
    return len(sub), len(miss), miss[:20]

# 랜덤하게 하나 찍어서 확인
one = train_df[["site_id","year"]].drop_duplicates().sample(1, random_state=0).iloc[0]
nrows, nmiss, miss20 = missing_doys_one_group(train_df, one["site_id"], one["year"])
print("example group:", one.to_dict(), "rows:", nrows, "missing:", nmiss, "miss20:", miss20)


In [ ]:
# site-year별 Transformer 입력 텐서 X[T,d] 만들기
def build_samples(df, feature_cols, T=365):
    import numpy as np
    samples = []
    for (site, year), sub in df.groupby(["site_id","year"], sort=False):
        sub = sub.sort_values("doy")

        # ✅ 윤년 제거: DOY 366은 버림
        sub = sub[sub["doy"] <= T].copy()

        X = sub[feature_cols].to_numpy(dtype=np.float32)

        if X.shape[0] != T:
            raise ValueError(f"{site}-{year}: got T={X.shape[0]} rows (expected {T}). Check DOY grid / missing days.")

        L = int(sub["L_doy"].iloc[0])
        R = int(sub["R_doy"].iloc[0])
        ctype = str(sub["censor_type"].iloc[0])

        # 라벨도 1..T로 클램프
        L = min(max(L, 1), T)
        R = min(max(R, 1), T)

        samples.append({
            "site_id": site,
            "year": int(year),
            "X": X,
            "L": L,
            "R": R,
            "censor_type": ctype
        })
    return samples


In [ ]:
samples = build_samples(train_df2, feature_cols, T=365)
train_samples, val_samples, test_samples = split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42)
x_mean, x_std = compute_norm_stats(train_samples)


from collections import Counter
print("train X lengths:", Counter([s["X"].shape[0] for s in train_samples]))
print("val   X lengths:", Counter([s["X"].shape[0] for s in val_samples]))
print("test  X lengths:", Counter([s["X"].shape[0] for s in test_samples]))
print("sizes:", len(train_samples), len(val_samples), len(test_samples))

In [ ]:
# 시즌 윈도우로 변경해서(예: DOY 60~300) 텐서 만들기

DOY_START, DOY_END = 60, 300
train_df_season = train_df2[(train_df2["doy"] >= DOY_START) & (train_df2["doy"] <= DOY_END)].copy()

# ✅ 라벨도 시즌 범위로 클램프 (중요)
train_df_season["L_doy"] = train_df_season["L_doy"].clip(lower=DOY_START, upper=DOY_END)
train_df_season["R_doy"] = train_df_season["R_doy"].clip(lower=DOY_START, upper=DOY_END)

# ✅ 시즌 길이 T
T = DOY_END - DOY_START + 1
print("Season T =", T)

# ✅ 이제 samples 만들기 (build_samples를 시즌용으로 약간 수정)
def build_samples_season(df, feature_cols, DOY_START=60, DOY_END=300):
    T = DOY_END - DOY_START + 1
    samples = []
    for (site, year), sub in df.groupby(["site_id","year"], sort=False):
        sub = sub.sort_values("doy")
        # 시즌 구간이 정확히 T개인지 확인
        if len(sub) != T:
            # 혹시 빠진 day가 있으면 에러 (현재는 missing=0이라 OK일 가능성 큼)
            continue

        X = sub[feature_cols].to_numpy(dtype=np.float32)

        # 라벨을 시즌 좌표(1..T)로 변환
        L = int(sub["L_doy"].iloc[0]) - DOY_START + 1
        R = int(sub["R_doy"].iloc[0]) - DOY_START + 1
        ctype = str(sub["censor_type"].iloc[0])

        # clamp
        L = min(max(L, 1), T)
        R = min(max(R, 1), T)

        samples.append({"site_id": site, "year": int(year), "X": X, "L": L, "R": R, "censor_type": ctype})
    return samples

samples = build_samples_season(train_df_season, feature_cols, DOY_START, DOY_END)
print("num samples(season):", len(samples))


In [ ]:
from collections import Counter
print(Counter([s["X"].shape[0] for s in train_samples]))


In [ ]:
print(len(train_samples), len(val_samples), len(test_samples), "total:", len(train_samples)+len(val_samples)+len(test_samples))


In [ ]:
# split + norm + DataLoader 만들기
# 1) split (site 기준)
train_samples, val_samples, test_samples = split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42)
print("split:", len(train_samples), len(val_samples), len(test_samples))

# 2) norm stats
x_mean, x_std = compute_norm_stats(train_samples)
print("x_mean:", x_mean)
print("x_std :", x_std)

# 3) Dataset / Loader
train_ds = IntervalEventDataset(train_samples, x_mean, x_std)
val_ds   = IntervalEventDataset(val_samples, x_mean, x_std)
test_ds  = IntervalEventDataset(test_samples, x_mean, x_std)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False, drop_last=False)

# sanity check
print("one batch shapes:", next(iter(train_loader))[0].shape)  # (B, T=241, D)
print("T =", T)


In [ ]:
# 정의
# =========================
# Step7-min: 정의 + 데이터 준비까지만 (학습 루프 없음)
# =========================
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- 1) split (site 기준) ----
def split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    sites = sorted(list({s["site_id"] for s in samples}))
    rng.shuffle(sites)

    n = len(sites)
    n_test = int(n * test_frac)
    n_val  = int(n * val_frac)

    test_sites  = set(sites[:n_test])
    val_sites   = set(sites[n_test:n_test+n_val])
    train_sites = set(sites[n_test+n_val:])

    train = [s for s in samples if s["site_id"] in train_sites]
    val   = [s for s in samples if s["site_id"] in val_sites]
    test  = [s for s in samples if s["site_id"] in test_sites]
    return train, val, test

train_samples, val_samples, test_samples = split_by_site(samples, 0.1, 0.1, seed=42)
print("split:", len(train_samples), len(val_samples), len(test_samples))

# ---- 2) norm stats (train만) ----
def compute_norm_stats(samples):
    X_all = np.concatenate([s["X"][None, :, :] for s in samples], axis=0)  # (N,T,D)
    mean = X_all.reshape(-1, X_all.shape[-1]).mean(axis=0)
    std  = X_all.reshape(-1, X_all.shape[-1]).std(axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)

x_mean, x_std = compute_norm_stats(train_samples)

# ---- 3) Dataset (L,R,ctype 텐서로) ----
CTYPE2ID = {"interval": 0, "right": 1, "left": 2}

class IntervalEventDataset(Dataset):
    def __init__(self, samples, mean, std):
        self.samples = samples
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        X = (s["X"] - self.mean) / self.std
        X = torch.from_numpy(X).float()  # (T,D)
        L = torch.tensor(int(s["L"]), dtype=torch.long)
        R = torch.tensor(int(s["R"]), dtype=torch.long)
        c = torch.tensor(CTYPE2ID[str(s["censor_type"])], dtype=torch.long)
        return X, L, R, c

train_ds = IntervalEventDataset(train_samples, x_mean, x_std)
val_ds   = IntervalEventDataset(val_samples,   x_mean, x_std)
test_ds  = IntervalEventDataset(test_samples,  x_mean, x_std)

# ---- 4) Model 정의 ----
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=400):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class HazardTransformer(nn.Module):
    def __init__(self, d_in, d_model=64, nhead=4, num_layers=3, dropout=0.2, max_len=400):
        super().__init__()
        self.in_proj = nn.Linear(d_in, d_model)
        self.pos = PositionalEncoding(d_model, max_len=max_len)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, X):
        h = self.in_proj(X)
        h = self.pos(h)
        h = self.encoder(h)
        logits = self.head(h).squeeze(-1)          # (B,T)
        return torch.sigmoid(logits).clamp(1e-6, 1-1e-6)

# ---- 5) 시즌 길이/입력차원 ----
T = int(train_samples[0]["X"].shape[0])   # 시즌이면 241, 연중이면 365
print("T =", T, "| D =", int(train_samples[0]['X'].shape[1]))


In [ ]:
# 모델 학습 + early stopping (generalization-friendly 세팅)
# =========================
# 모델 학습 + early stopping (seed 고정 포함 완전본)
# =========================
import copy
import random
import numpy as np
import torch
from torch.utils.data import DataLoader

# ---------- 0) SEED 고정 ----------
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---------- 1) DataLoader (shuffle 재현성 고정) ----------
g = torch.Generator()
g.manual_seed(SEED)

# train_ds/val_ds/test_ds가 이미 만들어져 있다고 가정
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False, generator=g)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False, drop_last=False)

#  interval_nll_per_sample
def interval_nll_per_sample(hazard, L, R, ctype, Tend):
    """
    return nll per sample: shape (B,)
    hazard: (B,T)
    L,R,ctype: (B,)
    """
    B, T = hazard.shape
    assert T == Tend

    log_surv_terms = torch.log1p(-hazard)      # (B,T)
    logS = torch.cumsum(log_surv_terms, dim=1) # log S_t

    L = torch.clamp(L, 1, Tend)
    R = torch.clamp(R, 1, Tend)
    idxL = (L - 1).long()
    idxR = (R - 1).long()

    logS_L = logS.gather(1, idxL.view(-1,1)).squeeze(1)
    logS_R = logS.gather(1, idxR.view(-1,1)).squeeze(1)

    nll = torch.zeros(B, device=hazard.device)

    mask_interval = (ctype == 0)
    mask_right    = (ctype == 1)
    mask_left     = (ctype == 2)

    # interval: -log(S_L - S_R)
    if mask_interval.any():
        a = logS_L[mask_interval]
        b = logS_R[mask_interval]
        a = torch.maximum(a, b + 1e-8)
        nll[mask_interval] = -(a + torch.log1p(-torch.exp(b - a)))

    # right: -log(S_T)
    if mask_right.any():
        nll[mask_right] = -logS[:, -1][mask_right]

    # left: -log(1 - S_R)
    if mask_left.any():
        a = logS_R[mask_left]
        cutoff = -0.6931471805599453
        out = torch.empty_like(a)
        m = a < cutoff
        out[m] = torch.log1p(-torch.exp(a[m]))
        out[~m]= torch.log(-torch.expm1(a[~m]))
        nll[mask_left] = -out

    return nll

# ---------- 2) train/eval 함수 ----------
W_INTERVAL = 1.0
W_LEFT = 0.5
W_RIGHT = 0.5

def weighted_loss_from_ctype(nll_vec, ctype):
    w = torch.ones_like(nll_vec)
    w = torch.where(ctype==0, torch.tensor(W_INTERVAL, device=nll_vec.device), w)
    w = torch.where(ctype==1, torch.tensor(W_RIGHT, device=nll_vec.device), w)
    w = torch.where(ctype==2, torch.tensor(W_LEFT, device=nll_vec.device), w)
    return (w * nll_vec).mean()

def run_epoch_weighted(model, opt, loader, Tend, train=True):
    model.train(train)
    total, n = 0.0, 0
    for X, L, R, ctype in loader:
        X = X.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)

        hazard = model(X)
        nll_vec = interval_nll_per_sample(hazard, L, R, ctype, Tend=Tend)
        loss = weighted_loss_from_ctype(nll_vec, ctype)

        if train:
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)


@torch.no_grad()
def eval_nll_model(model, loader, Tend):
    model.eval()
    total, n = 0.0, 0
    for X, L, R, ctype in loader:
        X = X.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)
        hazard = model(X)
        nll_vec = interval_nll_per_sample(hazard, L, R, ctype, Tend=Tend)
        loss = weighted_loss_from_ctype(nll_vec, ctype)

        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)

# ---------- 3) 모델 생성 (초기화 재현성 고정) ----------
torch.manual_seed(SEED)  # 모델 init도 고정
model = HazardTransformer(
    d_in=train_samples[0]["X"].shape[1],
    d_model=64,
    nhead=4,
    num_layers=3,
    dropout=0.2
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# ---------- 4) Early stopping 설정 ----------
best_val = float("inf")
best_state = None
best_epoch = -1

patience = 6      # 4보다 조금 늘려도 좋음
max_epochs = 60   # 30보다 늘려서 early stop이 제대로 작동하게
min_delta = 1e-3  # 너무 미세한 개선은 "개선"으로 안 치기

pat = 0

for epoch in range(1, max_epochs + 1):
    tr = run_epoch_weighted(model, opt, train_loader, Tend=T, train=True)
    va = eval_nll_model(model, val_loader, Tend=T)
    print(f"epoch {epoch:02d} | train_nll {tr:.4f} | val_nll {va:.4f}")

    if va < best_val - min_delta:
        best_val = va
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        pat = 0
    else:
        pat += 1
        if pat >= patience:
            print(f"[Early stop] best_epoch={best_epoch}, best_val_nll={best_val:.4f}")
            break

# ---------- 5) best 로드 ----------
if best_state is None:
    raise ValueError("best_state is None (no improvement captured). Try lowering min_delta or increasing max_epochs.")

model.load_state_dict(best_state)
print("best_epoch:", best_epoch, "best_val_nll:", best_val)

# ---------- 6) (선택) test 성능도 바로 찍기 ----------
test_nll = eval_nll_model(model, test_loader, Tend=T)
print("test_nll:", test_nll)



In [ ]:
# (1) point coverage/MAE, (2) mass-in-interval, (3) 80% 예측구간 overlap(IoU)
import numpy as np
import torch

CTYPE_INTERVAL = 0  # interval=0, right=1, left=2

@torch.no_grad()
def eval_nll_model(model, loader, Tend):
    model.eval()
    total, n = 0.0, 0
    for X, L, R, ctype in loader:
        X = X.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)
        hazard = model(X)
        loss = interval_nll(hazard, L, R, ctype, Tend=Tend)
        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)

@torch.no_grad()
def hazard_to_pmf_cdf_logS(hazard):
    B, T = hazard.shape
    logS = torch.cumsum(torch.log1p(-hazard), dim=1)  # log S_t
    S_prev = torch.cat([torch.ones(B,1,device=hazard.device), torch.exp(logS[:, :-1])], dim=1)
    pmf = S_prev * hazard
    cdf = torch.cumsum(pmf, dim=1).clamp(0, 1)
    return pmf, cdf, logS

def quantile_from_cdf_1d(cdf_1d, q, Tend):
    # cdf_1d: numpy (T,)
    if cdf_1d[-1] < q:        # 분포가 q까지 못 올라가면
        return Tend
    return int(np.searchsorted(cdf_1d, q) + 1)  # 1..T

def overlap_metrics(pred_L, pred_R, true_L, true_R):
    # pred interval: [pred_L, pred_R] inclusive
    # true interval: (L,R] -> [L+1, R] inclusive
    true_L2 = true_L + 1

    inter_L = max(pred_L, true_L2)
    inter_R = min(pred_R, true_R)
    inter = max(0, inter_R - inter_L + 1)

    pred_len = max(1, pred_R - pred_L + 1)
    true_len = max(1, true_R - true_L)  # = R-L
    union_L = min(pred_L, true_L2)
    union_R = max(pred_R, true_R)
    union = max(1, union_R - union_L + 1)

    iou = inter / union
    recall = inter / true_len
    precision = inter / pred_len
    return iou, recall, precision

@torch.no_grad()
def eval_metrics_with_overlap(model, loader, Tend, alpha=0.2):
    """
    alpha=0.2 => 중앙 80% 예측구간 [q0.1, q0.9]
    """
    model.eval()
    q_lo = alpha/2
    q_hi = 1 - alpha/2

    hits_all, maes_all, mass_all = [], [], []

    # interval-only overlap
    ious, recalls, precs = [], [], []
    hits_int, maes_int, mass_int = [], [], []
    n_int = 0

    for X, L, R, ctype in loader:
        X = X.to(device)
        L_t = L.to(device)
        R_t = R.to(device)

        ctype_np = ctype.cpu().numpy().astype(int)
        L_np = L.cpu().numpy().astype(int)
        R_np = R.cpu().numpy().astype(int)

        hazard = model(X)  # (B,T)
        pmf, cdf, logS = hazard_to_pmf_cdf_logS(hazard)

        # ---- median (CDF가 0.5 못 넘으면 Tend로) ----
        cdf_last = cdf[:, -1]
        arg = (cdf >= 0.5).float().argmax(dim=1) + 1
        median = torch.where(cdf_last >= 0.5, arg, torch.tensor(Tend, device=cdf.device))
        median_np = median.cpu().numpy().astype(int)

        # ---- point coverage & MAE(midpoint) ----
        hit = ((median_np > L_np) & (median_np <= R_np)).astype(float)
        mid = np.round((L_np + R_np) / 2.0).astype(int)
        mae = np.abs(median_np - mid).astype(float)

        hits_all.extend(hit.tolist())
        maes_all.extend(mae.tolist())

        # ---- mass-in-interval: S_L - S_R ----
        idxL = (torch.clamp(L_t, 1, Tend) - 1).long()
        idxR = (torch.clamp(R_t, 1, Tend) - 1).long()
        logS_L = logS.gather(1, idxL.view(-1,1)).squeeze(1)
        logS_R = logS.gather(1, idxR.view(-1,1)).squeeze(1)
        mass = (torch.exp(logS_L) - torch.exp(logS_R)).clamp(min=0.0).cpu().numpy()
        mass_all.extend(mass.tolist())

        # ---- overlap (interval-only) ----
        cdf_np = cdf.cpu().numpy()
        for b in range(len(L_np)):
            if ctype_np[b] != CTYPE_INTERVAL:
                continue
            n_int += 1
            hits_int.append(hit[b]); maes_int.append(mae[b]); mass_int.append(mass[b])

            pL = quantile_from_cdf_1d(cdf_np[b], q_lo, Tend)
            pR = quantile_from_cdf_1d(cdf_np[b], q_hi, Tend)
            pL = max(1, min(pL, Tend))
            pR = max(1, min(pR, Tend))
            if pL > pR:
                pL, pR = pR, pL

            iou, rec, prec = overlap_metrics(pL, pR, int(L_np[b]), int(R_np[b]))
            ious.append(iou); recalls.append(rec); precs.append(prec)

    out = {
        "point_cov_mean_all": float(np.mean(hits_all)),
        "mae_mid_mean_all": float(np.mean(maes_all)),
        "mass_in_interval_mean_all": float(np.mean(mass_all)),
        "mass_in_interval_median_all": float(np.median(mass_all)),

        "point_cov_mean_interval_only": float(np.mean(hits_int)) if n_int>0 else np.nan,
        "mae_mid_mean_interval_only": float(np.mean(maes_int)) if n_int>0 else np.nan,
        "mass_in_interval_mean_interval_only": float(np.mean(mass_int)) if n_int>0 else np.nan,

        "IoU_mean_interval_only(80%)": float(np.mean(ious)) if n_int>0 else np.nan,
        "Recall_mean_interval_only(80%)": float(np.mean(recalls)) if n_int>0 else np.nan,
        "Precision_mean_interval_only(80%)": float(np.mean(precs)) if n_int>0 else np.nan,
        "N_interval_samples": int(n_int),
    }
    return out

# ---- 실행 ----
val_nll = eval_nll_model(model, val_loader, Tend=T)
test_nll = eval_nll_model(model, test_loader, Tend=T)

val_stats = eval_metrics_with_overlap(model, val_loader, Tend=T, alpha=0.2)
test_stats = eval_metrics_with_overlap(model, test_loader, Tend=T, alpha=0.2)

print("=== FINAL EVAL ===")
print("VAL  NLL:", val_nll)
print("TEST NLL:", test_nll)

print("\n--- VAL stats ---")
for k, v in val_stats.items():
    print(k, ":", v)

print("\n--- TEST stats ---")
for k, v in test_stats.items():
    print(k, ":", v)



In [ ]:
alphas = [0.5, 0.2, 0.1]  # 50%, 80%, 90% predictive interval
rows = []

for a in alphas:
    stats = eval_metrics_with_overlap(model, test_loader, Tend=T, alpha=a)
    rows.append({
        "alpha": a,
        "credible_interval": f"{int((1-a)*100)}%",
        "point_cov_all": stats["point_cov_mean_all"],
        "mass_mean_all": stats["mass_in_interval_mean_all"],
        "IoU(only interval)": stats["IoU_mean_interval_only(80%)"] if "IoU_mean_interval_only(80%)" in stats else stats.get("IoU_mean_interval_only(80%)"),
        "Recall(only interval)": stats["Recall_mean_interval_only(80%)"] if "Recall_mean_interval_only(80%)" in stats else stats.get("Recall_mean_interval_only(80%)"),
        "Precision(only interval)": stats["Precision_mean_interval_only(80%)"] if "Precision_mean_interval_only(80%)" in stats else stats.get("Precision_mean_interval_only(80%)"),
        "N_interval": stats["N_interval_samples"],
    })

df = pd.DataFrame(rows)
print(df)


In [ ]:
from collections import Counter
import pandas as pd
import numpy as np

def split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    sites = sorted(list({s["site_id"] for s in samples}))
    rng.shuffle(sites)

    n = len(sites)
    n_test = int(n * test_frac)
    n_val  = int(n * val_frac)

    test_sites = set(sites[:n_test])
    val_sites  = set(sites[n_test:n_test+n_val])
    train_sites= set(sites[n_test+n_val:])

    train = [s for s in samples if s["site_id"] in train_sites]
    val   = [s for s in samples if s["site_id"] in val_sites]
    test  = [s for s in samples if s["site_id"] in test_sites]
    return train, val, test

train_s, val_s, test_s = split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42)

def censor_ratio(split_samples):
    c = Counter([s["censor_type"] for s in split_samples])
    total = sum(c.values())
    return {k: (v, v/total) for k, v in c.items()}, total

for name, split in [("train", train_s), ("val", val_s), ("test", test_s)]:
    dist, total = censor_ratio(split)
    print(f"\n[{name}] N={total}")
    for k,(v,p) in dist.items():
        print(f"  {k}: {v} ({p:.3f})")


In [ ]:
# left-censored 데이터 처리
# 관측 프로세스 feature 만들기: first_obs_doy, n_obs, max_gap
# A) obs2로 site-year별 관측 메타 만들기
import pandas as pd
import numpy as np

DOY_START, DOY_END = 60, 300

# obs2는 ["site_id","year","obs_doy","(트랩)복숭아순나방-마리수"] 형태라고 가정
obs2_season = obs2[(obs2["obs_doy"] >= DOY_START) & (obs2["obs_doy"] <= DOY_END)].copy()
obs2_season = obs2_season.sort_values(["site_id","year","obs_doy"])

def make_obs_meta(df):
    rows = []
    for (site, year), sub in df.groupby(["site_id","year"], sort=False):
        d = sub["obs_doy"].to_numpy()
        d = np.sort(d)
        n_obs = len(d)
        first_obs = int(d[0])
        # 관측 간격 최대
        if n_obs >= 2:
            max_gap = int(np.max(np.diff(d)))
        else:
            max_gap = int(DOY_END - DOY_START)  # 관측 1번뿐이면 매우 불확실하다고 크게
        rows.append({
            "site_id": site,
            "year": int(year),
            "first_obs_doy": first_obs,
            "n_obs": int(n_obs),
            "max_gap": int(max_gap),
        })
    return pd.DataFrame(rows)

obs_meta = make_obs_meta(obs2_season)

# 시즌 좌표(1..T)로도 만들어두면 모델에 더 직관적임
T = DOY_END - DOY_START + 1
obs_meta["first_obs_season"] = (obs_meta["first_obs_doy"] - DOY_START + 1).clip(1, T)

print(obs_meta.head())
print(obs_meta[["n_obs","max_gap","first_obs_doy"]].describe())


In [ ]:
# B) 이 메타를 train_df2에 merge해서 “매일 feature”로 붙이기
# train_df2 생성 이후(보간 끝난 뒤) 여기에 추가
train_df2 = train_df2.merge(obs_meta[["site_id","year","first_obs_season","n_obs","max_gap"]],
                            on=["site_id","year"], how="left")

# 관측메타 없는 경우(해당 시즌에 관측 자체가 없던 케이스)는 큰 불확실로 채우기
train_df2["first_obs_season"] = train_df2["first_obs_season"].fillna(1)   # 시즌 첫날로
train_df2["n_obs"] = train_df2["n_obs"].fillna(0)
train_df2["max_gap"] = train_df2["max_gap"].fillna(T)

# 숫자형으로
train_df2["first_obs_season"] = train_df2["first_obs_season"].astype(int)
train_df2["n_obs"] = train_df2["n_obs"].astype(int)
train_df2["max_gap"] = train_df2["max_gap"].astype(int)


In [ ]:
# C) feature_cols에 추가 
feature_cols = feature_cols + ["first_obs_season", "n_obs", "max_gap"]
print("new D_in =", len(feature_cols))

In [ ]:
# interval 얼마나 심한지 확인하기

DOY_START, DOY_END = 60, 300

# labels: site_id, year, censor_type, L_doy, R_doy 있다고 가정
interval = labels[labels["censor_type"] == "interval"].copy()

# 원래 gap
interval["gap_raw"] = interval["R_doy"] - interval["L_doy"]

# 시즌 범위로 클램프한 gap (우리가 학습에 쓰는 범위 기준)
interval["L_cl"] = interval["L_doy"].clip(DOY_START, DOY_END)
interval["R_cl"] = interval["R_doy"].clip(DOY_START, DOY_END)
interval["gap_clipped"] = interval["R_cl"] - interval["L_cl"]

print("N interval:", len(interval))
print("max gap_raw:", interval["gap_raw"].max())
print("max gap_clipped:", interval["gap_clipped"].max())

print("\nsummary (gap_clipped):")
print(interval["gap_clipped"].describe())

print("\nTop 10 largest gaps (clipped):")
print(interval.sort_values("gap_clipped", ascending=False)[
    ["site_id","year","L_doy","R_doy","gap_raw","gap_clipped"]
].head(10))



In [ ]:
# interval 길이 큰 애들 제거 (labels 단계에서)
DOY_START, DOY_END = 60, 300
T = DOY_END - DOY_START + 1

MAX_GAP = 30   # 먼저 30으로 시작 (원하면 14/21로도 테스트)

# labels: site_id, year, censor_type, L_doy, R_doy 있다고 가정
labels_f = labels.copy()

# 시즌 범위로 클램프해서 gap 계산
labels_f["L_cl"] = labels_f["L_doy"].clip(DOY_START, DOY_END)
labels_f["R_cl"] = labels_f["R_doy"].clip(DOY_START, DOY_END)
labels_f["gap"]  = labels_f["R_cl"] - labels_f["L_cl"]

print("interval gap max(clipped):", labels_f.loc[labels_f["censor_type"]=="interval", "gap"].max())

# interval만 필터링 (right/left는 그대로 유지)
before = len(labels_f)
labels_f = labels_f[(labels_f["censor_type"] != "interval") | (labels_f["gap"] <= MAX_GAP)].copy()
after = len(labels_f)

print(f"labels kept: {after}/{before}  (dropped {before-after})")
print(labels_f["censor_type"].value_counts())

labels = labels_f.drop(columns=["L_cl","R_cl","gap"], errors="ignore")


In [ ]:
# site static(lat/lon) 추가 (train_df2 단계에서)
site_static = (
    obs[["site_id", "좌표-위도", "좌표-경도"]]
    .drop_duplicates("site_id")
    .copy()
)


site_static["좌표-위도"] = pd.to_numeric(site_static["좌표-위도"], errors="coerce")
site_static["좌표-경도"] = pd.to_numeric(site_static["좌표-경도"], errors="coerce")


In [ ]:
train_df2 = train_df2.merge(site_static, on="site_id", how="left")

# 결측이면 전체 평균으로(간단 처리)
train_df2["좌표-위도"] = train_df2["좌표-위도"].fillna(train_df2["좌표-위도"].mean())
train_df2["좌표-경도"] = train_df2["좌표-경도"].fillna(train_df2["좌표-경도"].mean())

feature_cols = feature_cols + ["좌표-위도", "좌표-경도"]
print("new D_in =", len(feature_cols))

# --------

In [ ]:
# 기본 지표(NLL/COV/MAE) 뽑기
@torch.no_grad()
def predict_event_day(hazard):
    B, T_ = hazard.shape
    logS = torch.cumsum(torch.log1p(-hazard), dim=1)
    S_prev = torch.cat([torch.ones(B,1,device=hazard.device), torch.exp(logS[:, :-1])], dim=1)
    pmf = S_prev * hazard
    cdf = torch.cumsum(pmf, dim=1).clamp(0,1)
    median = (cdf >= 0.5).float().argmax(dim=1) + 1
    return pmf, cdf, median

@torch.no_grad()
def eval_cov_mae(model, loader):
    model.eval()
    hit, total = 0, 0
    maes = []
    for X, L, R, ctype in loader:
        X = X.to(device)
        hazard = model(X)
        _, _, median = predict_event_day(hazard)

        pred = median.cpu().numpy()
        L = L.numpy(); R = R.numpy()
        hit += ((pred > L) & (pred <= R)).sum()
        total += len(pred)

        mid = np.round((L + R)/2.0).astype(int)
        maes.extend(np.abs(pred - mid).tolist())

    return hit/max(total,1), float(np.mean(maes))

val_nll = eval_nll_model(model, val_loader, Tend=T)
test_nll = eval_nll_model(model, test_loader, Tend=T)
val_cov, val_mae = eval_cov_mae(model, val_loader)
test_cov, test_mae = eval_cov_mae(model, test_loader)

print("VAL : NLL", val_nll, "cov", val_cov, "mae", val_mae)
print("TEST: NLL", test_nll, "cov", test_cov, "mae", test_mae)


In [ ]:
# Transformer hazard 모델 + interval-censored NLL 학습 (PyTorch)
# =========================
# Step 7-0. imports
# =========================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# samples: Step6에서 만든 list[dict]를 그대로 사용한다고 가정
# samples[i] = {"site_id", "year", "X"(365,8), "L", "R", "censor_type"}

# =========================
# Step 7-1. Split (cross-location: site 기준 분할)
# =========================
def split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=0):
    rng = np.random.default_rng(seed)
    sites = sorted(list({s["site_id"] for s in samples}))
    rng.shuffle(sites)

    n = len(sites)
    n_test = int(n * test_frac)
    n_val  = int(n * val_frac)

    test_sites = set(sites[:n_test])
    val_sites  = set(sites[n_test:n_test+n_val])
    train_sites= set(sites[n_test+n_val:])

    train = [s for s in samples if s["site_id"] in train_sites]
    val   = [s for s in samples if s["site_id"] in val_sites]
    test  = [s for s in samples if s["site_id"] in test_sites]
    return train, val, test

train_samples, val_samples, test_samples = split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42)
print(len(train_samples), len(val_samples), len(test_samples))

# =========================
# Step 7-2. Feature normalization (train 통계로 표준화)
# =========================
def compute_norm_stats(samples):
    # (N, T, D)로 모아서 mean/std (D별)
    X_all = np.concatenate([s["X"][None, :, :] for s in samples], axis=0)  # (N,T,D)
    mean = X_all.reshape(-1, X_all.shape[-1]).mean(axis=0)
    std  = X_all.reshape(-1, X_all.shape[-1]).std(axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)

x_mean, x_std = compute_norm_stats(train_samples)
print("mean:", x_mean)
print("std :", x_std)

# =========================
# Step 7-3. Dataset / DataLoader
# =========================
CTYPE2ID = {"interval": 0, "right": 1, "left": 2}

class IntervalEventDataset(Dataset):
    def __init__(self, samples, mean, std):
        self.samples = samples
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        X = (s["X"] - self.mean) / self.std  # (365, D)
        X = torch.from_numpy(X).float()

        # DOY는 1..365. 그대로 사용
        L = int(s["L"])
        R = int(s["R"])
        ctype = CTYPE2ID[str(s["censor_type"])]

        return X, torch.tensor(L), torch.tensor(R), torch.tensor(ctype)

train_ds = IntervalEventDataset(train_samples, x_mean, x_std)
val_ds   = IntervalEventDataset(val_samples, x_mean, x_std)
test_ds  = IntervalEventDataset(test_samples, x_mean, x_std)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False, drop_last=False)

# =========================
# Step 7-4. Model: Transformer encoder + hazard head
# =========================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=400):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        T = x.size(1)
        return x + self.pe[:, :T, :]

class HazardTransformer(nn.Module):
    def __init__(self, d_in, d_model=128, nhead=4, num_layers=3, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(d_in, d_model)
        self.pos = PositionalEncoding(d_model, max_len=400)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.hazard_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, X):
        # X: (B, T, D)
        h = self.in_proj(X)
        h = self.pos(h)
        h = self.encoder(h)
        logits = self.hazard_head(h).squeeze(-1)     # (B, T)
        hazard = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
        return hazard  # (B, T)

model = HazardTransformer(d_in=train_samples[0]["X"].shape[1], d_model=128, nhead=4, num_layers=3, dropout=0.1).to(device)
print(model)

# =========================
# Step 7-5. Interval-censored NLL loss (안정적 구현)
# =========================
def log1mexp(a):
    # log(1-exp(a)) for a <= 0, stable
    # a is (B,)
    # from: https://cran.r-project.org/web/packages/Rmpfr/vignettes/log1mexp-note.pdf
    out = torch.empty_like(a)
    cutoff = -0.6931471805599453  # log(0.5)
    mask = a < cutoff
    out[mask] = torch.log1p(-torch.exp(a[mask]))
    out[~mask]= torch.log(-torch.expm1(a[~mask]))
    return out

def logdiffexp(a, b):
    # log(exp(a) - exp(b)), stable, requires a > b
    # a,b: (B,)
    return a + torch.log1p(-torch.exp(b - a))

def interval_nll(hazard, L, R, ctype, Tend=365):
    """
    hazard: (B,T) with T=365. hazard[t-1] corresponds to day t.
    L,R: (B,) DOY in [1..365]
    ctype: (B,) {0:interval,1:right,2:left}
    """

    B, T = hazard.shape
    assert T == Tend

    # logS_t = sum_{k=1..t} log(1-h_k)
    log_surv_terms = torch.log1p(-hazard)           # (B,T)
    logS = torch.cumsum(log_surv_terms, dim=1)      # logS[:, t-1] = log S_t
    # convenience: S_0 = 1 -> logS0 = 0
    # S_L: survival after day L -> logS_L = logS[:, L-1]
    # S_R: survival after day R -> logS_R = logS[:, R-1]

    # clamp indices
    L = torch.clamp(L, 1, Tend)
    R = torch.clamp(R, 1, Tend)

    idxL = (L - 1).long()
    idxR = (R - 1).long()

    logS_L = logS.gather(1, idxL.view(-1,1)).squeeze(1)  # (B,)
    logS_R = logS.gather(1, idxR.view(-1,1)).squeeze(1)  # (B,)

    # interval: -log(S_L - S_R)  (S_L > S_R)
    # right:   -log(S_Tend)
    # left:    -log(1 - S_R)

    nll = torch.zeros(B, device=hazard.device)

    mask_interval = (ctype == 0)
    mask_right    = (ctype == 1)
    mask_left     = (ctype == 2)

    if mask_interval.any():
        a = logS_L[mask_interval]
        b = logS_R[mask_interval]
        # ensure a > b numerically; if L==R then prob ~0 -> huge loss
        # small epsilon safeguard
        a = torch.maximum(a, b + 1e-8)
        nll[mask_interval] = -logdiffexp(a, b)

    if mask_right.any():
        logS_T = logS[:, -1][mask_right]
        nll[mask_right] = -logS_T

    if mask_left.any():
        # -log(1 - S_R) = -log(1 - exp(logS_R))
        a = logS_R[mask_left]  # <= 0
        nll[mask_left] = -log1mexp(a)

    return nll.mean()

# =========================
# Step 7-6. Train loop
# =========================
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

def run_epoch(loader, train=True):
    model.train(train)
    total = 0.0
    n = 0
    for X, L, R, ctype in loader:
        X = X.to(device)
        L = L.to(device)
        R = R.to(device)
        ctype = ctype.to(device)

        hazard = model(X)
        loss = interval_nll(hazard, L, R, ctype, Tend=T)

        if train:
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)

for epoch in range(1, 11):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    print(f"epoch {epoch:02d} | train_nll {tr:.4f} | val_nll {va:.4f}")



In [ ]:
# Test NLL 계산 (메인 지표)
@torch.no_grad()
def eval_nll(loader, Tend):
    model.eval()
    total = 0.0
    n = 0
    for X, L, R, ctype in loader:
        X = X.to(device)
        L = L.to(device)
        R = R.to(device)
        ctype = ctype.to(device)

        hazard = model(X)
        loss = interval_nll(hazard, L, R, ctype, Tend=Tend)

        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)


In [ ]:
# 예측 분포 → median day (시즌 좌표 1..T)
@torch.no_grad()
def predict_event_day(hazard):
    """
    hazard: (B,T)
    return median (1..T) in season coordinates
    """
    B, T = hazard.shape
    logS = torch.cumsum(torch.log1p(-hazard), dim=1)  # log S_t
    S_prev = torch.cat([torch.ones(B,1,device=hazard.device), torch.exp(logS[:, :-1])], dim=1)  # S_{t-1}
    pmf = S_prev * hazard
    cdf = torch.cumsum(pmf, dim=1).clamp(0,1)
    median = (cdf >= 0.5).float().argmax(dim=1) + 1  # 1..T
    return pmf, cdf, median


In [ ]:
# coverage: median이 (L,R] 안에 들어가는 비율 (시즌 좌표)
@torch.no_grad()
def eval_coverage(loader):
    model.eval()
    hit = 0
    total = 0
    for X, L, R, ctype in loader:
        X = X.to(device)
        hazard = model(X)
        _, _, median = predict_event_day(hazard)

        pred = median.cpu().numpy()
        L = L.numpy()
        R = R.numpy()

        ok = (pred > L) & (pred <= R)   # (L,R] 정의
        hit += ok.sum()
        total += len(pred)
    return hit / max(total, 1)


In [ ]:
# MAE(midpoint) (시즌 좌표 기준)
@torch.no_grad()
def eval_mae_midpoint(loader):
    model.eval()
    maes = []
    for X, L, R, ctype in loader:
        X = X.to(device)
        hazard = model(X)
        _, _, median = predict_event_day(hazard)

        pred = median.cpu().numpy()
        L = L.numpy()
        R = R.numpy()
        mid = np.round((L + R) / 2.0).astype(int)

        maes.extend(np.abs(pred - mid).tolist())
    return float(np.mean(maes))


In [ ]:
print("=== SEASON SUMMARY ===")
print("T =", T, "DOY_START =", DOY_START, "DOY_END =", DOY_END)

print("val NLL :", eval_nll(val_loader, Tend=T))
print("test NLL:", eval_nll(test_loader, Tend=T))

print("val coverage:", eval_coverage(val_loader))
print("test coverage:", eval_coverage(test_loader))

print("val MAE(midpoint) [season coords]:", eval_mae_midpoint(val_loader))
print("test MAE(midpoint) [season coords]:", eval_mae_midpoint(test_loader))


In [ ]:
THRESHOLDS = [1, 2, 3]
results = []

for th in THRESHOLDS:
    print("\n======================")
    print("THRESHOLD =", th)
    
    # 1) obs2 -> labels
    labels = build_interval_labels_from_doy_fixed(obs2, threshold=th, count_col="(트랩)복숭아순나방-마리수")
    
    # 2) daily + labels merge
    train_df = daily_feat.merge(labels, on=["site_id","year"], how="inner")
    
    # 3) weather 보간 (이미 쓰던 함수)
    train_df2 = fill_weather_nan_by_group(train_df, weather_cols)
    
    # 4) season slice
    DOY_START, DOY_END = 60, 300
    train_df_season = train_df2[(train_df2["doy"] >= DOY_START) & (train_df2["doy"] <= DOY_END)].copy()
    train_df_season["L_doy"] = train_df_season["L_doy"].clip(DOY_START, DOY_END)
    train_df_season["R_doy"] = train_df_season["R_doy"].clip(DOY_START, DOY_END)
    
    # 5) samples 생성
    samples = build_samples_season(train_df_season, feature_cols, DOY_START, DOY_END)
    T = DOY_END - DOY_START + 1
    
    # 6) split / norm / dataloader
    train_samples, val_samples, test_samples = split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42)
    x_mean, x_std = compute_norm_stats(train_samples)
    
    train_ds = IntervalEventDataset(train_samples, x_mean, x_std)
    val_ds   = IntervalEventDataset(val_samples, x_mean, x_std)
    test_ds  = IntervalEventDataset(test_samples, x_mean, x_std)
    
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False)
    
    # 7) 모델 새로 만들고 학습 (epoch는 일단 6~8만 돌려도 충분)
    model = HazardTransformer(d_in=train_samples[0]["X"].shape[1], d_model=128, nhead=4, num_layers=3, dropout=0.1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

    for epoch in range(1, 9):
        tr = run_epoch(train_loader, train=True)    # 내부에서 model/opt 사용하도록 되어있다면 수정 필요
        va = run_epoch(val_loader, train=False)
        print(f"epoch {epoch:02d} | train_nll {tr:.4f} | val_nll {va:.4f}")

    # 8) 지표
    val_nll = eval_nll(val_loader, Tend=T)
    test_nll = eval_nll(test_loader, Tend=T)
    val_cov = eval_coverage(val_loader)
    test_cov = eval_coverage(test_loader)
    val_mae = eval_mae_midpoint(val_loader)
    test_mae = eval_mae_midpoint(test_loader)
    
    print("VAL: nll", val_nll, "cov", val_cov, "mae", val_mae)
    print("TEST:nll", test_nll, "cov", test_cov, "mae", test_mae)
    
    results.append([th, val_nll, test_nll, val_cov, test_cov, val_mae, test_mae])

print("\n=== SUMMARY TABLE ===")
import pandas as pd
res_df = pd.DataFrame(results, columns=["threshold","val_nll","test_nll","val_cov","test_cov","val_mae","test_mae"])
print(res_df)


In [ ]:
import copy
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# =========================
# 0) 설정
# =========================
TH = 1
DOY_START, DOY_END = 60, 300
T = DOY_END - DOY_START + 1

# SAVE_PATH = "best_hazard_transformer_th1_season.pt"  # 원하는 경로로 바꿔도 됨

# =========================
# 1) TH=1 labels 재생성 (obs2 기반)
# =========================
labels = build_interval_labels_from_doy_fixed(
    obs2, threshold=TH, count_col="(트랩)복숭아순나방-마리수"
)

# daily + labels merge
train_df = daily_feat.merge(labels, on=["site_id","year"], how="inner")

# 기상 보간
train_df2 = fill_weather_nan_by_group(train_df, weather_cols)

# season slice
train_df_season = train_df2[(train_df2["doy"] >= DOY_START) & (train_df2["doy"] <= DOY_END)].copy()
train_df_season["L_doy"] = train_df_season["L_doy"].clip(DOY_START, DOY_END)
train_df_season["R_doy"] = train_df_season["R_doy"].clip(DOY_START, DOY_END)

# samples (L/R는 시즌 좌표로 변환됨)
samples = build_samples_season(train_df_season, feature_cols, DOY_START, DOY_END)
print("num samples:", len(samples), "T:", T)

# split
train_samples, val_samples, test_samples = split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=42)
print("split:", len(train_samples), len(val_samples), len(test_samples))

# norm stats
x_mean, x_std = compute_norm_stats(train_samples)

# dataloaders
train_ds = IntervalEventDataset(train_samples, x_mean, x_std)
val_ds   = IntervalEventDataset(val_samples, x_mean, x_std)
test_ds  = IntervalEventDataset(test_samples, x_mean, x_std)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False, drop_last=False)

# =========================
# 2) run_epoch (model/opt 인자 받는 안전 버전)
# =========================
def run_epoch(model, opt, loader, Tend, train=True):
    model.train(train)
    total = 0.0
    n = 0
    for X, L, R, ctype in loader:
        X = X.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)
        hazard = model(X)
        loss = interval_nll(hazard, L, R, ctype, Tend=Tend)

        if train:
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        total += loss.item() * X.size(0)
        n += X.size(0)

    return total / max(n, 1)

@torch.no_grad()
def eval_nll_model(model, loader, Tend):
    model.eval()
    total = 0.0
    n = 0
    for X, L, R, ctype in loader:
        X = X.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)
        hazard = model(X)
        loss = interval_nll(hazard, L, R, ctype, Tend=Tend)
        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)

# predict median day (season coords 1..T)
@torch.no_grad()
def predict_event_day(hazard):
    B, T_ = hazard.shape
    logS = torch.cumsum(torch.log1p(-hazard), dim=1)
    S_prev = torch.cat([torch.ones(B,1,device=hazard.device), torch.exp(logS[:, :-1])], dim=1)
    pmf = S_prev * hazard
    cdf = torch.cumsum(pmf, dim=1).clamp(0,1)
    median = (cdf >= 0.5).float().argmax(dim=1) + 1
    return pmf, cdf, median

@torch.no_grad()
def eval_coverage_model(model, loader):
    model.eval()
    hit, total = 0, 0
    for X, L, R, ctype in loader:
        X = X.to(device)
        hazard = model(X)
        _, _, median = predict_event_day(hazard)
        pred = median.cpu().numpy()
        L = L.numpy(); R = R.numpy()
        ok = (pred > L) & (pred <= R)
        hit += ok.sum()
        total += len(pred)
    return hit / max(total, 1)

@torch.no_grad()
def eval_mae_midpoint_model(model, loader):
    model.eval()
    maes = []
    for X, L, R, ctype in loader:
        X = X.to(device)
        hazard = model(X)
        _, _, median = predict_event_day(hazard)
        pred = median.cpu().numpy()
        L = L.numpy(); R = R.numpy()
        mid = np.round((L + R) / 2.0).astype(int)
        maes.extend(np.abs(pred - mid).tolist())
    return float(np.mean(maes))

# =========================
# 3) 모델 학습 + early stopping
# =========================
model = HazardTransformer(
    d_in=train_samples[0]["X"].shape[1],
    d_model=64,      # 128 -> 64
    nhead=4,
    num_layers=3,
    dropout=0.2      # 0.1 -> 0.2
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)  # 2e-4 -> 1e-4


best_val = float("inf")
best_epoch = -1
best_state = None

patience = 4
pat = 0
max_epochs = 30

for epoch in range(1, max_epochs+1):
    tr = run_epoch(model, opt, train_loader, Tend=T, train=True)
    va = eval_nll_model(model, val_loader, Tend=T)
    print(f"epoch {epoch:02d} | train_nll {tr:.4f} | val_nll {va:.4f}")

    if va < best_val - 1e-4:
        best_val = va
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        pat = 0
    else:
        pat += 1
        if pat >= patience:
            print(f"[Early stop] best_epoch={best_epoch}, best_val_nll={best_val:.4f}")
            break

# best 로드
model.load_state_dict(best_state)

# =========================
# 4) Best 모델로 최종 평가 + 저장
# =========================
val_nll  = eval_nll_model(model, val_loader, Tend=T)
test_nll = eval_nll_model(model, test_loader, Tend=T)
val_cov  = eval_coverage_model(model, val_loader)
test_cov = eval_coverage_model(model, test_loader)
val_mae  = eval_mae_midpoint_model(model, val_loader)
test_mae = eval_mae_midpoint_model(model, test_loader)

print("\n=== BEST SUMMARY (TH=1) ===")
print("best_epoch:", best_epoch)
print("val  NLL:", val_nll, "cov:", val_cov, "mae:", val_mae)
print("test NLL:", test_nll, "cov:", test_cov, "mae:", test_mae)

ckpt = {
    "state_dict": model.state_dict(),
    "x_mean": x_mean,
    "x_std": x_std,
    "feature_cols": feature_cols,
    "threshold": TH,
    "DOY_START": DOY_START,
    "DOY_END": DOY_END,
    "T": T,
    "best_epoch": best_epoch,
    "metrics": {
        "val_nll": val_nll, "test_nll": test_nll,
        "val_cov": val_cov, "test_cov": test_cov,
        "val_mae": val_mae, "test_mae": test_mae
    }
}
#torch.save(ckpt, SAVE_PATH)
#print("saved:", SAVE_PATH)


In [ ]:
# 기존 feature(8개) vs 추가 feature(10개)
import copy
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

TH = 1
DOY_START, DOY_END = 60, 300
T = DOY_END - DOY_START + 1
SEED = 42

def ffill_bfill_group(df, group_cols, cols):
    df = df.sort_values(group_cols + ["doy"]).copy()
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df[c] = df.groupby(group_cols)[c].transform(lambda s: s.ffill().bfill())
    return df


# ---- 평가 함수들 (모델 인자로 받는 버전) ----
def run_epoch(model, opt, loader, Tend, train=True):
    model.train(train)
    total = 0.0
    n = 0
    for X, L, R, ctype in loader:
        X = X.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)
        hazard = model(X)
        loss = interval_nll(hazard, L, R, ctype, Tend=Tend)

        if train:
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)

@torch.no_grad()
def eval_nll_model(model, loader, Tend):
    model.eval()
    total = 0.0
    n = 0
    for X, L, R, ctype in loader:
        X = X.to(device); L = L.to(device); R = R.to(device); ctype = ctype.to(device)
        hazard = model(X)
        loss = interval_nll(hazard, L, R, ctype, Tend=Tend)
        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / max(n, 1)

@torch.no_grad()
def predict_median_day(model, X):
    # X: torch tensor (B,T,D)
    hazard = model(X)
    B, T_ = hazard.shape
    logS = torch.cumsum(torch.log1p(-hazard), dim=1)
    S_prev = torch.cat([torch.ones(B,1,device=hazard.device), torch.exp(logS[:, :-1])], dim=1)
    pmf = S_prev * hazard
    cdf = torch.cumsum(pmf, dim=1).clamp(0,1)
    median = (cdf >= 0.5).float().argmax(dim=1) + 1
    return median  # 1..T (season coords)

@torch.no_grad()
def eval_coverage_model(model, loader):
    model.eval()
    hit = 0
    total = 0
    for X, L, R, ctype in loader:
        X = X.to(device)
        median = predict_median_day(model, X).cpu().numpy()
        L = L.numpy(); R = R.numpy()
        ok = (median > L) & (median <= R)
        hit += ok.sum()
        total += len(median)
    return hit / max(total, 1)

@torch.no_grad()
def eval_mae_midpoint_model(model, loader):
    model.eval()
    maes = []
    for X, L, R, ctype in loader:
        X = X.to(device)
        median = predict_median_day(model, X).cpu().numpy()
        L = L.numpy(); R = R.numpy()
        mid = np.round((L + R) / 2.0).astype(int)
        maes.extend(np.abs(median - mid).tolist())
    return float(np.mean(maes))


# ---- 실험 1회 실행 함수 ----
def run_experiment(feature_mode: str, feature_cols_base: list):
    """
    feature_mode:
      - "base8": 기존 8개
      - "plus10": 기존 8개 + tmean_7d + rain_7d
    """
    # 1) labels 생성 (TH=1 고정)
    labels = build_interval_labels_from_doy_fixed(obs2, threshold=TH, count_col="(트랩)복숭아순나방-마리수")

    # 2) merge
    train_df = daily_feat.merge(labels, on=["site_id","year"], how="inner")

    # 3) 결측 보간
    train_df2 = fill_weather_nan_by_group(train_df, weather_cols)
    train_df2 = train_df2.sort_values(["site_id","year","doy"]).copy()
    train_df2["평균기온(°C)"] = pd.to_numeric(train_df2["평균기온(°C)"], errors="coerce")
    train_df2["일강수량(mm)"] = pd.to_numeric(train_df2["일강수량(mm)"], errors="coerce")


    # 4) rolling feature 추가(plus10만)
    if feature_mode == "plus10":
        train_df2["tmean_7d"] = (
            train_df2.groupby(["site_id","year"])["평균기온(°C)"]
            .transform(lambda s: s.rolling(window=7, min_periods=1).mean())
        )
        train_df2["rain_7d"] = (
            train_df2.groupby(["site_id","year"])["일강수량(mm)"]
            .transform(lambda s: s.rolling(window=7, min_periods=1).sum())
        )
        feature_cols = feature_cols_base + ["tmean_7d", "rain_7d"]
    else:
        feature_cols = feature_cols_base

    train_df2 = ffill_bfill_group(train_df2, ["site_id","year"], feature_cols)
    train_df2[feature_cols] = train_df2[feature_cols].fillna(0.0)


    # 5) season slice
    train_df_season = train_df2[(train_df2["doy"] >= DOY_START) & (train_df2["doy"] <= DOY_END)].copy()
    train_df_season["L_doy"] = train_df_season["L_doy"].clip(DOY_START, DOY_END)
    train_df_season["R_doy"] = train_df_season["R_doy"].clip(DOY_START, DOY_END)

    # 6) samples
    samples = build_samples_season(train_df_season, feature_cols, DOY_START, DOY_END)

    # 7) split / norm / loaders
    train_samples, val_samples, test_samples = split_by_site(samples, val_frac=0.1, test_frac=0.1, seed=SEED)
    x_mean, x_std = compute_norm_stats(train_samples)

    train_ds = IntervalEventDataset(train_samples, x_mean, x_std)
    val_ds   = IntervalEventDataset(val_samples, x_mean, x_std)
    test_ds  = IntervalEventDataset(test_samples, x_mean, x_std)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False, drop_last=False)

    # 8) 모델/옵티마이저 (generalization-friendly setting)
    d_in = train_samples[0]["X"].shape[1]
    model = HazardTransformer(d_in=d_in, d_model=64, nhead=4, num_layers=3, dropout=0.2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    # 9) early stopping
    best_val = float("inf")
    best_state = None
    best_epoch = -1
    patience = 4
    pat = 0
    max_epochs = 30

    for epoch in range(1, max_epochs+1):
        tr = run_epoch(model, opt, train_loader, Tend=T, train=True)
        va = eval_nll_model(model, val_loader, Tend=T)

        if va < best_val - 1e-4:
            best_val = va
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    model.load_state_dict(best_state)

    # 10) 최종 지표
    val_nll = eval_nll_model(model, val_loader, Tend=T)
    test_nll = eval_nll_model(model, test_loader, Tend=T)
    val_cov = eval_coverage_model(model, val_loader)
    test_cov = eval_coverage_model(model, test_loader)
    val_mae = eval_mae_midpoint_model(model, val_loader)
    test_mae = eval_mae_midpoint_model(model, test_loader)

    return {
        "feature_set": feature_mode,
        "d_in": d_in,
        "best_epoch": best_epoch,
        "val_nll": float(val_nll),
        "test_nll": float(test_nll),
        "val_cov": float(val_cov),
        "test_cov": float(test_cov),
        "val_mae": float(val_mae),
        "test_mae": float(test_mae),
    }


# ---- 실행: base8 vs plus10 ----
feature_cols_base = [
    "일강수량(mm)",
    "최고기온(°C)",
    "최저기온(°C)",
    "평균기온(°C)",
    "평균 풍속(m/s)",
    "최대 풍속(m/s)",
    "DD10",
    "GDD10_cum",
]

res1 = run_experiment("base8", feature_cols_base)
print("DONE base8:", res1)

res2 = run_experiment("plus10", feature_cols_base)
print("DONE plus10:", res2)

res_df = pd.DataFrame([res1, res2]).sort_values("test_nll")
print("\n=== COMPARISON ===")
print(res_df)
